Differential expression analysis of Bulk RNA-seq data of MEF cells


Date last updated: January 23,2026

Created by : Sayane Shome

The workflow for the DESEQ2 analysis has been adapted from the tutorial :
https://rstudio-pubs-static.s3.amazonaws.com/329027_593046fb6d7a427da6b2c538caf601e1.html#example-3-two-conditions-two-genotypes-with-an-interaction-term


"Here we have two genotypes, wild-type (wt) and mutant (mut). Two conditions: control (Control) and treated (LPS treated). We are interested in the responses of both wild-type and mutant to treatment. We are also interested in the differences in response between genotypes, which is captured by the interaction term in linear models."

The order of the samples has to be WT +/- LPS and then KO +/- LPS.

In [ ]:
version #to make sure which R version and computer architecture the code was run

In [ ]:
#loading packages

In [ ]:
library(EnsDb.Mmusculus.v79)
library(DESeq2)
library(EnhancedVolcano)
library(tidyr)
library(ashr)
library(tidyverse)
library(knitr)
library(Glimma)
library(DESeq2)
library(pheatmap)
library(ggplot2)
library(dplyr)
library(plotly)
library(VennDiagram)
library(clusterProfiler)
library(enrichplot)
library(ggplot2)
library(ggridges)


In [ ]:
#loading count matrix from processed data. The count matrix contains the count values obtained from RSEM. 

s <- read.csv('all_unnorm_counts_MEF_minus_het_arranged.csv')
colnames(s)[1] <- "Ensemble_ID"
s4 <- s
#reading metadata about the count matrix
coldata <- read.csv('MEF_counts/coldata_minus_het_mef2020counts_v1.csv',header = FALSE)
metadata <- coldata



Complete sets of LPS + Ctrl

P4_5_lps1,P4_5_lps2,P4_5_lps3,P7_4_lps1,P7_4_lps2,P8_4_lps3,P8_4_lps4,P8_4_lps5,P4_5_ctrl1,P4_5_ctrl2,P4_5_ctrl3,P7_4_ctrl1,P7_4_ctrl2,P8_4_ctrl3,P8_4_ctrl4,P8_4_ctrl5	

In [ ]:
#renaming columns

colnames(metadata)[1] <- "sample"
colnames(metadata)[2] <- "id"
colnames(metadata)[3] <- "genotype"
colnames(metadata)[4] <- "condition"


In [ ]:
#This is how metadata table looks like.
metadata

In [ ]:
# Filtering rows with zero counts in more than four columns/samples

In [ ]:
zeros_count <- apply(s4 == 0, 1, sum)
filtered_df <- s4[zeros_count <= 4, ]
s4 <- filtered_df 
s4[,3:18] <- s4[,3:18]+1 
s4[,3:18] <- round(s4[,3:18])

In [ ]:
countData <- s4     #assigning metadata and countdata
metaData <- metadata


In [ ]:
#Formatted the countData dataframe and 

In [ ]:
countData <- read.csv('all_unnorm_counts_MEF_minus_het_arranged_genenames_added_for_ensembleids_v1.csv')

In [ ]:
rownames(countData) <- countData$Ensemble_ID
#countData$Ensemble_ID <- NULL

In [ ]:
countData <- countData %>%
  mutate(Gene_name = ifelse(is.na(Gene_name) | Gene_name == "", Ensemble_ID, Gene_name))

In [ ]:
# Assuming numeric columns 2 to n contain expression data
countData_resolved <- countData %>%
  group_by(Gene_name) %>%
  summarise(across(where(is.numeric), mean))

In [ ]:
countData_resolved

In [ ]:
countData1 <- as.data.frame(countData_resolved)
countData1[,2:17] <- round(countData1[,2:17])

In [ ]:
countData1

In [ ]:
new_order <- c(9,10,11,12,13,1,2,3,4,5,14,15,16,6,7,8)

# Step 3: Rearrange the dataframe by the specified row indices
metadata <- metadata[new_order, ]
metadata

In [ ]:
metadata

In [ ]:
library(DESeq2)
library(tibble) # For tidy data handling

# Assuming 'metadata' is your metadata DataFrame and 'countData1' is your count matrix
# First, make sure 'countData1' has the correct row names (genes) and column names (samples)
rownames(countData1) <- countData1$Gene_name
countData1 <- countData1[, -1] # Remove the 'Gene_name' column

# Make sure the 'metadata' DataFrame is correctly ordered to match the samples in 'countData1'
# The order should match exactly with the column names of 'countData1'
metadata_ordered <- metadata[match(colnames(countData1), metadata$id), ]

# Create a DESeqDataSet
# Assuming 'metadata_ordered' now has the columns in the correct order: 'sample', 'id', 'genotype', 'condition'
# The 'design' formula you have can stay the same
dds <- DESeqDataSetFromMatrix(countData = countData1,
                              colData = DataFrame(metadata_ordered),
                              design = ~ genotype + condition + genotype:condition)

# At this point, 'dds' should have 'genotype' and 'condition' correctly aligned
# You can proceed with your DESeq2 analysis workflow


In [ ]:
#Setting reference as Control and wildtype(wt) respectively.

In [ ]:
#dds$condition <- factor(dds$condition, levels = c("LPS","Control"))
#dds$genotype <- factor(dds$genotype, levels = c("wt","mut"))

dds$condition <- relevel(dds$condition, ref = "Control")
dds$genotype <- relevel(dds$genotype, ref = "wt")

In [ ]:
#running DESEq to obtain the DE results

In [ ]:
#design(dds) <- ~ genotype + condition + genotype:condition
dds <- DESeq(dds) 
resultsNames(dds)

In [ ]:
#Next part of code shows the results of how many DE genes shows up.

In [ ]:
normalized_counts <- counts(dds, normalized=TRUE)
write.csv(normalized_counts,"MEF_Deseq2_2020countsdata_normalized_chk.csv")

In [ ]:
manual_annotation <- data.frame(
  Genotype = factor(c("wt", "wt", "wt","wt","wt","wt","wt","wt","wt","wt","mut","mut","mut","mut","mut","mut")),
  Condition = factor(c("Control", "Control", "Control", "Control","Control","LPS", "LPS", "LPS","LPS","LPS","Control","Control", "Control","LPS","LPS","LPS"))
)

#annotation_col = as.data.frame(manual_annotation)

ann_colors <- list(
  Genotype = c("wt" = "#fe9388", "mut" = "#9bca1e"),
  Condition = c("Control" = "#05d8e1", "LPS" = "#e298fe")
)


# Ensure row names of annotations match the sample names, if needed
rownames(manual_annotation) <- colnames(normalized_counts)

In [ ]:
row_means_norm_counts <- apply(normalized_counts, 1, mean)

ordered_indices_norm_counts <- order(-row_means_norm_counts)
ordered_norm_counts <- normalized_counts[ordered_indices_norm_counts, ]
rows_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", rownames(ordered_norm_counts))

# Filter the dataframe to remove undesired rows
ordered_norm_counts <- ordered_norm_counts[rows_to_keep, ]

# Assuming ordered_norm_counts, manual_annotation, and ann_colors are already defined

# Define the color palette, you might adjust this based on your preference
color_palette <- colorRampPalette(c("blue", "white", "red"))(101) # 101 colors for -3 to 3 scale

# Generate breaks from -3 to 3
breaks <- seq(-3, 3, length.out = length(color_palette))

# Plot the heatmap with the specified breaks and color scale
pheatmap(ordered_norm_counts[1:50, ],
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "row", 
         show_rownames = TRUE, 
         show_colnames = FALSE,
         annotation_col = manual_annotation,
         annotation_colors = ann_colors,
         breaks = breaks,
         color = color_palette)

# Automatically added: save last plot as SVG
ggsave('figure_033.svg', width = 8, height = 6)


In [ ]:
# ---------- Order & filter ----------
row_means_norm_counts <- apply(normalized_counts, 1, mean)

ordered_indices_norm_counts <- order(-row_means_norm_counts)
ordered_norm_counts <- normalized_counts[ordered_indices_norm_counts, ]

rows_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", 
                       rownames(ordered_norm_counts))

ordered_norm_counts <- ordered_norm_counts[rows_to_keep, ]

# ---------- Color scale ----------
color_palette <- colorRampPalette(c("blue", "white", "red"))(101)
breaks <- seq(-3, 3, length.out = length(color_palette))

# ---------- Plot to screen (optional, just to see) ----------
pheatmap(
  ordered_norm_counts[1:50, ],
  cluster_rows   = FALSE,
  cluster_cols   = FALSE,
  scale          = "row",
  show_rownames  = TRUE,
  show_colnames  = FALSE,
  annotation_col = manual_annotation,
  annotation_colors = ann_colors,
  breaks         = breaks,
  color          = color_palette
)

# ---------- Save as SVG using svglite backend ----------
ggplot2::ggsave(
  filename = "figure_033_test.svg",
  width    = 8,
  height   = 6,
  device   = svglite::svglite    # force svglite device
)

cat("Saved SVG to:", normalizePath("figure_033_test.svg"), "\n")


In [ ]:
# -----------------------------
# 1. Inspect working directory
# -----------------------------
cat("Current working directory:\n", getwd(), "\n\n")

# -----------------------------
# 2. Recompute & filter matrix
# -----------------------------
row_means_norm_counts <- apply(normalized_counts, 1, mean)

ordered_indices_norm_counts <- order(-row_means_norm_counts)
ordered_norm_counts <- normalized_counts[ordered_indices_norm_counts, ]

rows_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", 
                       rownames(ordered_norm_counts))

ordered_norm_counts <- ordered_norm_counts[rows_to_keep, ]

# -----------------------------
# 3. Color scale
# -----------------------------
color_palette <- colorRampPalette(c("blue", "white", "red"))(101)
breaks <- seq(-3, 3, length.out = length(color_palette))

# -----------------------------
# 4. Output file in *this* folder
# -----------------------------
out_file <- "figure_033_test.svg"

# Very important: load pheatmap
if (!"pheatmap" %in% installed.packages()[,"Package"]) {
  install.packages("pheatmap")
}
library(pheatmap)

# -----------------------------
# 5. Open base SVG device
#    (no svglite/systemfonts)
# -----------------------------
svg(out_file, width = 8, height = 6)

pheatmap(
  ordered_norm_counts[1:50, ],
  cluster_rows   = FALSE,
  cluster_cols   = FALSE,
  scale          = "row",
  show_rownames  = TRUE,
  show_colnames  = FALSE,
  annotation_col = manual_annotation,
  annotation_colors = ann_colors,
  breaks         = breaks,
  color          = color_palette,
  use_raster    = FALSE  # <- FORCE VECTOR TILES
)

dev.off()  # close the SVG device

# -----------------------------
# 6. Verify file existence
# -----------------------------
cat("SVG path:\n", normalizePath(out_file), "\n")
cat("File exists? ", file.exists(out_file), "\n")

if (file.exists(out_file)) {
  print(file.info(out_file))
}


In [ ]:
# Make sure pheatmap is available
if (!"pheatmap" %in% installed.packages()[,"Package"]) {
  install.packages("pheatmap")
}
library(pheatmap)

save_top50_pheatmap_svg <- function(outfile = "figure_033_test.svg") {
  cat("Working directory:\n", getwd(), "\n")

  # 1) Prepare matrix
  row_means_norm_counts <- apply(normalized_counts, 1, mean)
  ordered_indices_norm_counts <- order(-row_means_norm_counts)
  ordered_norm_counts <- normalized_counts[ordered_indices_norm_counts, ]

  rows_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", 
                         rownames(ordered_norm_counts))
  ordered_norm_counts <- ordered_norm_counts[rows_to_keep, ]

  # 2) Colors
  color_palette <- colorRampPalette(c("blue", "white", "red"))(101)
  breaks <- seq(-3, 3, length.out = length(color_palette))

  cat("Devices before:\n")
  print(dev.list())

  # 3) Open SVG device
  svg(outfile, width = 8, height = 6)

  # Ensure device is always closed
  on.exit({
    dev.off()
    cat("Closed SVG device.\nDevices after closing:\n")
    print(dev.list())
    cat("File info:\n")
    print(file.info(outfile))
  }, add = TRUE)

  # 4) Try plotting
  ok <- tryCatch({
    pheatmap(
      ordered_norm_counts[1:50, ],
      cluster_rows      = FALSE,
      cluster_cols      = FALSE,
      scale             = "row",
      show_rownames     = TRUE,
      show_colnames     = FALSE,
      annotation_col    = manual_annotation,
      annotation_colors = ann_colors,
      breaks            = breaks,
      color             = color_palette,
      use_raster        = FALSE   # <-- force vector tiles
    )
    TRUE
  }, error = function(e) {
    message("Error in pheatmap: ", e$message)
    FALSE
  })

  if (!ok) {
    message("Heatmap failed; SVG may be empty.")
  } else {
    cat("Plotting completed.\n")
  }
}

# ---- Run it ----
save_top50_pheatmap_svg("figure_033_test.svg")


In [ ]:
# Make sure pheatmap is installed and loaded
if (!"pheatmap" %in% installed.packages()[, "Package"]) {
  install.packages("pheatmap")
}
library(pheatmap)

# 1) Recompute and filter matrix
row_means_norm_counts <- apply(normalized_counts, 1, mean)

ordered_indices_norm_counts <- order(-row_means_norm_counts)
ordered_norm_counts <- normalized_counts[ordered_indices_norm_counts, ]

rows_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", 
                       rownames(ordered_norm_counts))

ordered_norm_counts <- ordered_norm_counts[rows_to_keep, ]

# 2) Colors
color_palette <- colorRampPalette(c("blue", "white", "red"))(101)
breaks <- seq(-3, 3, length.out = length(color_palette))

# 3) Draw heatmap on the default device (RStudio Plots pane)
pheatmap(
  ordered_norm_counts[1:50, ],
  cluster_rows      = FALSE,
  cluster_cols      = FALSE,
  scale             = "row",
  show_rownames     = TRUE,
  show_colnames     = FALSE,
  annotation_col    = manual_annotation,
  annotation_colors = ann_colors,
  breaks            = breaks,
  color             = color_palette,
  use_raster        = FALSE   # again force vector tiles
)

# 4) Copy the current plot to SVG
out_file_copy <- "figure_033_copy.svg"

grDevices::dev.copy(
  grDevices::svg,
  filename = out_file_copy,
  width    = 8,
  height   = 6
)
dev.off()   # closes the SVG device we just created

# 5) Verify
cat("Copied SVG path:\n", normalizePath(out_file_copy), "\n")
print(file.info(out_file_copy))


In [ ]:
install.packages("svglite")   # Install once
library(svglite)              # Load it


In [ ]:
# Force using binary packages only for this session
options(install.packages.check.source = "no")

install.packages("systemfonts", type = "binary")
install.packages("svglite", type = "binary")

library(systemfonts)
library(svglite)
library(pheatmap)


In [ ]:
# Step 3: Plot the heatmap with ordered rows of normalized counts
pheatmap(ordered_norm_counts[1:50,],
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "none", 
         show_rownames = TRUE, 
         show_colnames = FALSE,
         annotation_col = manual_annotation,
         annotation_colors = ann_colors,# Adjust based on your preference
         color = colorRampPalette(c("blue", "white", "red"))(255)) # Adjust color gradient as needed
# Automatically added: save last plot as SVG
ggsave('figure_034.svg', width = 8, height = 6)


In [ ]:
LMC_top50genes_bycounts <- read.csv('LMC_top50_meanvalues_genenames.csv')

In [ ]:
LMC_top50genes_bycounts$X <- NULL
colnames(LMC_top50genes_bycounts)[1] <- "Gene_name"


In [ ]:
LMC_genes_data <- ordered_norm_counts[rownames(ordered_norm_counts) %in% LMC_top50genes_bycounts$Gene_name, ]
gene_names <- LMC_top50genes_bycounts$Gene_name[LMC_top50genes_bycounts$Gene_name %in% rownames(LMC_genes_data)]
LMC_genes_data_ordered <- LMC_genes_data[gene_names, ]


In [ ]:
pheatmap(LMC_genes_data_ordered,
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "none", 
         show_rownames = TRUE, 
         show_colnames = FALSE,
         annotation_col = manual_annotation,
         annotation_colors = ann_colors,# Adjust based on your preference
         color = colorRampPalette(c("blue", "white", "red"))(255)) # Adjust color gradient as needed
# Automatically added: save last plot as SVG
ggsave('figure_038.svg', width = 8, height = 6)


In [ ]:
breaks <- seq(-3, 3, length.out = length(color_palette))

# Plot the heatmap with the specified breaks and color scale
pheatmap(LMC_genes_data_ordered,
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "row", 
         show_rownames = TRUE, 
         show_colnames = FALSE,
         annotation_col = manual_annotation,
         annotation_colors = ann_colors,
         breaks = breaks,
         color = color_palette)
# Automatically added: save last plot as SVG
ggsave('figure_039.svg', width = 8, height = 6)


In [ ]:
groups <- c("wt_Control", "wt_Control", "wt_Control","wt_Control","wt_Control","wt_LPS","wt_LPS","wt_LPS","wt_LPS","wt_LPS","mut_Control","mut_Control","mut_Control", "mut_LPS", "mut_LPS","mut_LPS")

In [ ]:
normalized_counts_1 <- data.frame(normalized_counts)

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
})

###########################
## 1️⃣ Derive group information from column names
###########################
# Example assumption:
# Column names look like: "WT_Control_1", "WT_LPS_2", "MUT_Control_1", "MUT_LPS_3"
# Adjust the patterns below to match your real naming scheme.

colnames(expr_collapsed)

# Derive Genotype and Condition automatically
annotation_col <- data.frame(
  Genotype = ifelse(grepl("WT", colnames(expr_collapsed), ignore.case = TRUE), "WT",
                    ifelse(grepl("MUT", colnames(expr_collapsed), ignore.case = TRUE), "MUT", "Unknown")),
  Condition = ifelse(grepl("LPS", colnames(expr_collapsed), ignore.case = TRUE), "LPS",
                     ifelse(grepl("CTRL|CONTROL", colnames(expr_collapsed), ignore.case = TRUE), "Control", "Unknown"))
)

# Add row names (must match colnames of expr_collapsed)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 2️⃣ Define colors for annotation
###########################
ann_colors <- list(
  Genotype = c(WT = "#66c2a5", MUT = "#fc8d62", Unknown = "grey80"),
  Condition = c(Control = "#8da0cb", LPS = "#e78ac3", Unknown = "grey90")
)

###########################
## 3️⃣ Nicer heatmap (publication-quality)
###########################
hm_colors <- colorRampPalette(c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"))(200)

pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "row",
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 20,
  border_color = NA,
  main = "Heatmap of HOX/PAX/POU Gene Expression",
  treeheight_row = 10,
  treeheight_col = 15
)

# Automatically added: save last plot as SVG
ggsave('figure_042.svg', width = 8, height = 6)


In [ ]:
normalized_counts_2 <- data.frame(
  Gene = rownames(normalized_counts_1),
  P8_4_Lps3_minus_Ctrl3 = normalized_counts_1$P8_4_Lps3 - normalized_counts_1$P8_4_Ctrl3,
  P8_4_Lps5_minus_Ctrl5 = normalized_counts_1$P8_4_Lps5 - normalized_counts_1$P8_4_Ctrl5,
  P8_4_Lps4_minus_Ctrl4 = normalized_counts_1$P8_4_Lps4 - normalized_counts_1$P8_4_Ctrl4,
  P7_4_Lps1_minus_Ctrl1 = normalized_counts_1$P7_4_Lps1 - normalized_counts_1$P7_4_Ctrl1,
  P7_4_Lps2_minus_Ctrl2 = normalized_counts_1$P7_4_Lps2 - normalized_counts_1$P7_4_Ctrl2,
  P4_5_Lps2_minus_Ctrl2 = normalized_counts_1$P4_5_Lps2 - normalized_counts_1$P4_5_Ctrl2,
  P4_5_Lps3_minus_Ctrl3 = normalized_counts_1$P4_5_Lps3 - normalized_counts_1$P4_5_Ctrl3,
  P4_5_Lps1_minus_Ctrl1 = normalized_counts_1$P4_5_Lps1 - normalized_counts_1$P4_5_Ctrl1
)


In [ ]:
colnames(normalized_counts_2)[1] <- "Gene_name"
colnames(normalized_counts_2)[7] <- "P8_4_3"
colnames(normalized_counts_2)[9] <- "P8_4_5"
colnames(normalized_counts_2)[8] <- "P8_4_4"
colnames(normalized_counts_2)[5] <- "P7_4_1"
colnames(normalized_counts_2)[6] <- "P7_4_2"
colnames(normalized_counts_2)[3] <- "P4_5_2"
colnames(normalized_counts_2)[4] <- "P4_5_3"
colnames(normalized_counts_2)[2] <- "P4_5_1"

In [ ]:
glimmaMDS(dds,html = "MDS_plot_MEF_all_samples_March27_2024.html")

In [ ]:
rlog_data <- rlog(dds)
pca_data <- plotPCA(rlog_data, intgroup=c("condition", "genotype"), returnData=TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))
ggplot(pca_data, aes(x = PC1, y = PC2, color = condition, shape = genotype)) +
    geom_point(size = 3) +
    geom_text_repel(aes(label = row.names(pca_data))) +
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    ggtitle("PCA of RNA-seq samples") +
    theme_bw() +
    scale_shape_manual(values = c(15, 17, 19))

# Automatically added: save last plot as SVG
ggsave('figure_046.svg', width = 8, height = 6)


In [ ]:
library(DESeq2)
library(ggplot2)
library(ggrepel) # Ensure you have ggrepel installed for geom_text_repel

# Assuming rlog_data and dds are already defined and correctly prepared
rlog_data <- rlog(dds)

pca_data <- plotPCA(rlog_data, intgroup=c("condition", "genotype"), returnData=TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# Generate the PCA plot without sample names
ggplot(pca_data, aes(x = PC1, y = PC2, color = condition, shape = genotype)) +
    geom_point(size = 3) +
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    ggtitle("PCA of RNA-seq samples") +
    theme_bw() +
    scale_shape_manual(values = c(15, 17, 19)) # Adjust shapes or colors as needed

# Automatically added: save last plot as SVG
ggsave('figure_047.svg', width = 8, height = 6)


In [ ]:
pca_data_onlyControl <- pca_data %>%
  filter(condition == "Control")

In [ ]:
ggplot(pca_data_onlyControl, aes(x = PC1, y = PC2, color = condition, shape = genotype)) +
    geom_point(size = 4, stroke = 1) + # Adjust size and stroke for visibility
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    ggtitle("PCA of RNA-seq samples (Control condition)") +
    theme_bw() +
    scale_shape_manual(values = c(15, 17, 19)) # Adjust shapes or colors as needed

# Automatically added: save last plot as SVG
ggsave('figure_049.svg', width = 8, height = 6)


In [ ]:
ggplot(pca_data_onlyControl, aes(x = PC1, y = PC2, color = genotype)) +
    geom_point(size = 4, stroke = 1, shape = 16) + # Shape set to 16 for all points
    xlab(paste0("PC1: ", percentVar[1], "% variance")) +
    ylab(paste0("PC2: ", percentVar[2], "% variance")) +
    ggtitle("PCA of RNA-seq samples (Control condition)") +
    theme_bw() +
    scale_color_manual(values = c("wt" = "#96cb01", "mut" = "#fe9388")) # Adjust as per your genotypes
# Automatically added: save last plot as SVG
ggsave('figure_050.svg', width = 8, height = 6)


Carrying out DE analysis using DESeq2


Next part of code should help answer this :


I.The effect of treatment in wild-type.[This is for WT, treated compared with untreated.]

1.Genes upregulated with LPS in wild-type samples

2.Genes downregulated with LPS in wild-type samples



II.The effect of treatment in mutant[This is for knockout,treated compared with untreated ]   


3.Genes upregulated with LPS in Sp3 knockout samples

4.Genes downregulated with LPS in Sp3 knockout samples



III.What is the difference between mutant and wild-type in control treatments ?[mutant ctrl compared with wild type ctrl]

5.Genes upregulated in wild-type samples in control treatment/Genes downregulated in wild-type samples in control treatment

6.Genes downregulated in wild-type samples in control treatment/Genes downregulated in wild-type samples in control treatment



IV.With LPS treatment, what is the difference between mutant and wild-type?[mutant LPS compared with wild type LPS]

8.Genes upregulated in mutant samples compared to wildtype with LPS treatment

9.Genes downregulated in mutant samples compared to wildtype with LPS treatment




In [ ]:
res <- results(dds)
summary(res)

In [ ]:
#Plotting Glimma/scatter plots showing expression levels of particular gene across samples.
countData2 <- countData1
rownames(countData2) <- toupper(rownames(countData2))

gene_expression <- countData2["MEF2A",]  # Extract expression levels for particular gene
plot_data <- data.frame(
  Sample = colnames(countData2),  
  Expression = as.numeric(gene_expression),  
  Group = groups  
)
gene_name <- "MEF2A"

In [ ]:
# Create Genotype and Condition columns based on the Group column
plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group), "WT", "MUT")
plot_data$Condition <- ifelse(grepl("Control", plot_data$Group), "Control", "LPS")

# Create a new column for the combined labels
plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))

# Define the colors
genotype_colors <- c("WT" = "#fe9388", "MUT" = "#9bca1e")
condition_colors <- c("Control" = "#add8e6", "LPS" = "#e298fe")  # Lighter blue for Control

# Define the colors
genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")  # Adjusted colors
condition_colors <- c("Control" = "#B0DBFF", "LPS" = "#CF98E2")  # Lightened inside colors

# Create the plot
ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
  geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +  # Use shape 21 for filled points with outline
  scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +  # Inner circle colors and remove outline in legend
  scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +  # Outer circle colors
  scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +  # Shape 21 for circles, 22 for squares
  scale_x_discrete(limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")) +  # Set the order of the x-axis labels
  labs(
    title = paste("Expression of", gene_name, "Across Groups"),
    x = "",
    y = "Expression Level"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(size = 14, color = "black"),  # Increase x-axis label size
    axis.text.y = element_text(size = 14, color = "black"),  # Increase y-axis label size
    axis.title.x = element_text(size = 16),  # Increase x-axis title size
    axis.title.y = element_text(size = 16),  # Increase y-axis title size
    axis.ticks = element_line(color = "black"),
    axis.line = element_line(color = "black"),
    panel.background = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12),  # Increase legend text size
    legend.title = element_text(size = 14),  # Increase legend title size
    legend.key = element_blank()  # Remove legend key outline
  )
# Automatically added: save last plot as SVG
ggsave('figure_055.svg', width = 8, height = 6)


In [ ]:
library(dplyr)

# Calculate mean and sd for each group
summary_data <- plot_data %>%
  group_by(Group_Combined, Genotype, Condition) %>%
  summarise(
    mean_expression = mean(Expression, na.rm = TRUE),
    sd_expression = sd(Expression, na.rm = TRUE),
    n = n()
  )


In [ ]:
ggplot(summary_data, aes(x = Group_Combined, y = mean_expression)) +
  geom_point(
    aes(fill = Condition, color = Genotype, shape = Genotype),
    size = 9, stroke = 2
  ) +
  geom_errorbar(
    aes(ymin = mean_expression - sd_expression, ymax = mean_expression + sd_expression),
    width = 0.3, linewidth = 1
  ) +
  scale_fill_manual(
    name = "Condition",
    values = condition_colors,
    guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))
  ) +
  scale_color_manual(
    name = "Genotype",
    values = genotype_colors,
    labels = c("WT" = "Wild-type", "MUT" = "Mutant")
  ) +
  scale_shape_manual(
    name = "Genotype",
    values = c("WT" = 21, "MUT" = 22),
    labels = c("WT" = "Wild-type", "MUT" = "Mutant")
  ) +
  scale_x_discrete(
    limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")
  ) +
  labs(
    title = paste("Mean Expression of", gene_name, "Across Groups"),
    x = "",
    y = "Mean Expression ± SD"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(size = 14, color = "black"),
    axis.text.y = element_text(size = 14, color = "black"),
    axis.title.x = element_text(size = 16),
    axis.title.y = element_text(size = 16),
    axis.ticks = element_line(color = "black"),
    axis.line = element_line(color = "black"),
    panel.background = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12),
    legend.title = element_text(size = 14),
    legend.key = element_blank()
  )

# Automatically added: save last plot as SVG
ggsave('figure_057.svg', width = 8, height = 6)


In [ ]:
# Load required libraries
library(ggplot2)

# Ensure gene names are in uppercase
countData2 <- countData1
rownames(countData2) <- toupper(rownames(countData2))

# Define gene lists
gene_list <- c(
  "CCL4"
)

# Define colors
genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")
condition_colors <- c("Control" = "#B0DBFF", "LPS" = "#CF98E2")

# Create output directory
output_dir <- "Gene_Expression_Plots"
if (!dir.exists(output_dir)) dir.create(output_dir)

# Loop through each gene and generate the plots
for (gene in gene_list) {
  if (gene %in% rownames(countData2)) {
    gene_expression <- countData2[gene,]  # Extract expression levels
    
    plot_data <- data.frame(
      Sample = colnames(countData2),
      Expression = as.numeric(gene_expression),
      Group = groups
    )

    # Assign Genotype and Condition
    plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group, ignore.case = TRUE), "WT", "MUT")
    plot_data$Condition <- ifelse(grepl("Control", plot_data$Group, ignore.case = TRUE), "Control", "LPS")
    plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))

    # Generate plot
    p <- ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
      geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +
      scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +
      scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
      scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
      scale_x_discrete(limits = c("WT+Control", "WT+LPS", "MUT+Control", "MUT+LPS")) +
      labs(
        title = paste("Expression of", gene, "Across Groups"),
        x = "",
        y = "Expression Level"
      ) +
      theme_minimal() +
      theme(
        axis.text.x = element_text(size = 14, color = "black"),
        axis.text.y = element_text(size = 14, color = "black"),
        axis.title.x = element_text(size = 16),
        axis.title.y = element_text(size = 16),
        axis.ticks = element_line(color = "black"),
        axis.line = element_line(color = "black"),
        panel.background = element_blank(),
        panel.border = element_rect(color = "black", fill = NA),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        legend.position = "right",
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14),
        legend.key = element_blank()
      )

    # Save plot
    plot_filename <- file.path(output_dir, paste0(gene, "_mef_expression_plot.png"))
    ggsave(plot_filename, plot = p, width = 8, height = 6, dpi = 300)
    
    message("Saved plot for: ", gene)
  } else {
    message("Gene not found in data: ", gene)
  }
}

message("All plots saved in the directory: ", output_dir)

# Automatically added: save last plot as SVG
ggsave('figure_058.svg', width = 8, height = 6)


In [ ]:
# Load required libraries
library(ggplot2)

# Ensure gene names are in uppercase
countData2 <- countData1
rownames(countData2) <- toupper(rownames(countData2))

# Define gene list
gene_list <- c(
  "INCENP", "ZWILCH", "SPDL1", "CDCA8", "CDT1", "STIL", "GEN1", "SPC24", "TRIP13", "AURKA",
  "BUB1B", "AURKB", "BIRC5", "TTK", "NDC80", "CENPE", "CDC20", "NUF2", "PLK1", "BUB1",
  "KNTC1", "CCNB1",
  "PITX1", "TBX4", "HOXA11", "HOXA10", "BMP4", "SALL1", "HOXC11", "HOXC10", "LMX1B",
  "ALX3", "MSX1", "ALDH1A2", "GDF5", "GJA1", "LEF1", "NOG", "TBX2", "CRABP2", "HAND2",
  "CYP26B1", "TBX5", "RARB", "SHOX2", "FGF9", "HOXA9", "MSX2"
)

# Define colors
genotype_colors <- c("WT" = "#EC3A91", "MUT" = "#71980E")
condition_colors <- c("Control" = "#B0DBFF")  # Only Control remains

# Create output directory
output_dir <- "Gene_Expression_Plots"
if (!dir.exists(output_dir)) dir.create(output_dir)

# Loop through each gene and generate the plots
for (gene in gene_list) {
  if (gene %in% rownames(countData2)) {
    gene_expression <- countData2[gene,]  # Extract expression levels
    
    plot_data <- data.frame(
      Sample = colnames(countData2),
      Expression = as.numeric(gene_expression),
      Group = groups
    )

    # Assign Genotype and Condition
    plot_data$Genotype <- ifelse(grepl("wt", plot_data$Group, ignore.case = TRUE), "WT", "MUT")
    plot_data$Condition <- ifelse(grepl("Control", plot_data$Group, ignore.case = TRUE), "Control", "LPS")
    
    # Filter out LPS groups
    plot_data <- subset(plot_data, Condition == "Control")
    
    # Create a new column for the combined labels
    plot_data$Group_Combined <- with(plot_data, paste(Genotype, Condition, sep = "+"))

    # Generate plot
    p <- ggplot(plot_data, aes(x = Group_Combined, y = Expression)) +
      geom_point(aes(fill = Condition, color = Genotype, shape = Genotype), size = 9, stroke = 2) +
      scale_fill_manual(name = "Condition", values = condition_colors, guide = guide_legend(override.aes = list(shape = 21, size = 9, stroke = 0))) +
      scale_color_manual(name = "Genotype", values = genotype_colors, labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
      scale_shape_manual(name = "Genotype", values = c("WT" = 21, "MUT" = 22), labels = c("WT" = "Wild-type", "MUT" = "Mutant")) +
      scale_x_discrete(limits = c("WT+Control", "MUT+Control")) +  # Only control groups
      labs(
        title = paste("Expression of", gene, "Across Groups"),
        x = "",
        y = "Expression Level"
      ) +
      theme_minimal() +
      theme(
        axis.text.x = element_text(size = 14, color = "black"),
        axis.text.y = element_text(size = 14, color = "black"),
        axis.title.x = element_text(size = 16),
        axis.title.y = element_text(size = 16),
        axis.ticks = element_line(color = "black"),
        axis.line = element_line(color = "black"),
        panel.background = element_blank(),
        panel.border = element_rect(color = "black", fill = NA),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        legend.position = "right",
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14),
        legend.key = element_blank()
      )

    # Save plot
    plot_filename <- file.path(output_dir, paste0(gene, "_mef_expression_plot_withoutLPS.png"))
    ggsave(plot_filename, plot = p, width = 8, height = 6, dpi = 300)
    
    message("Saved plot for: ", gene)
  } else {
    message("Gene not found in data: ", gene)
  }
}

message("All plots saved in the directory: ", output_dir)

# Automatically added: save last plot as SVG
ggsave('figure_059.svg', width = 8, height = 6)


In [ ]:
library(ggplot2)
library(RColorBrewer)

# Assuming 'plot_data' is already prepared with 'Sample', 'Expression', 'Group', and 'GeneName'
gene_name <- "Nfkbie"

# Prepare colors
num_groups <- length(unique(plot_data$Group))
palette <- brewer.pal(num_groups, "Set1")
if (num_groups > 9) { # Set1 has a maximum of 9 colors
  palette <- colorRampPalette(brewer.pal(9, "Set1"))(num_groups)
}

# Order the 'Group' factor based on unique values
plot_data$Group <- factor(plot_data$Group, levels = unique(plot_data$Group))

# Plot with sample names
ggplot(plot_data, aes(x = Group, y = Expression, color = Group)) +
  geom_point(size = 5) +  # Increase the size of the points
  geom_text(aes(label = Sample), vjust = -0.5, size = 3) +  # Add text labels for each sample
  scale_color_manual(values = palette) +  # Use the generated palette for colors
  labs(
    title = paste("Expression of", gene_name, "Across Groups"),
    x = "Group",
    y = "Expression Level"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    axis.text.y = element_text(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.line = element_line(color = "black"),
    panel.background = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.position = "none"
  )

# Automatically added: save last plot as SVG
ggsave('figure_060.svg', width = 8, height = 6)


In [ ]:
res <- results(dds, name="genotypemut.conditionLPS",alpha = 0.05)
topGenes <- head(order(res$padj), 50)
topGeneCounts <- countData[topGenes,]


In [ ]:
topGenes <- head(order(res$log2FoldChange, na.last=NA), 60)  # na.last=NA removes NAs
countData <- counts(dds, normalized=TRUE)
topGeneCounts <- countData[topGenes, ]
rownames(manual_annotation) <- colnames(topGeneCounts)

In [ ]:
# Remove rows where row name starts with 'NA'
filtered_topGeneCounts <- topGeneCounts[!grepl("^NA", rownames(topGeneCounts)), ]


In [ ]:
# Filter the rows of filtered_topGeneCounts based on your criteria
filtered_rows <- rownames(filtered_topGeneCounts)[!grepl("^Gm|^Mir|^ENSMUS|^Rpl|Rik$", rownames(filtered_topGeneCounts))]

# Subset the dataframe to include only the filtered rows
filtered_topGeneCounts_filtered <- filtered_topGeneCounts[filtered_rows, ]

# Create the heatmap with the filtered dataframe
pheatmap(filtered_topGeneCounts_filtered, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE)
# Automatically added: save last plot as SVG
ggsave('figure_065.svg', width = 8, height = 6)


In [ ]:
# Create the heatmap with the filtered dataframe
pheatmap(filtered_topGeneCounts_filtered, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         scale = "row",
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE)
# Automatically added: save last plot as SVG
ggsave('figure_066.svg', width = 8, height = 6)


In [ ]:
cell_cycle_mef <- read.table('mef_cellcyclegenes.txt')
cell_cycle_lmc <- read.table('lmc_cellcyclegenes.txt')
colnames(cell_cycle_mef)[1] <- "Gene_name"
colnames(cell_cycle_lmc)[1] <- "Gene_name"
cell_cycle_mef_combo <- rbind(cell_cycle_lmc,cell_cycle_mef)

cell_cycle_mef_combo$Gene_name <- toupper(cell_cycle_mef_combo$Gene_name)







# Filter the countData to include only genes in the nf_kb list
rownames(countData) <- toupper(rownames(countData))
cell_cycle_res <- intersect(rownames(countData), cell_cycle_mef_combo$Gene_name)
filteredCountData_cellcycle <- countData[cell_cycle_res, ]



# Plot the heatmap for the top 100 genes
pheatmap(filteredCountData_cellcycle, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         annotation_col = as.data.frame(manual_annotation), 
         show_rownames = TRUE,scale = 'row')
# Automatically added: save last plot as SVG
ggsave('figure_067.svg', width = 8, height = 6)


In [ ]:
# Create the manual_annotation data frame with only Genotype
manual_annotation <- data.frame(
  Genotype = factor(c("wt", "wt", "wt", "wt", "wt", "wt", "wt", "wt", "wt", "wt", "mut", "mut", "mut", "mut", "mut", "mut")),
  Condition = factor(c("Control", "Control", "Control", "Control", "Control", "LPS", "LPS", "LPS", "LPS", "LPS", "Control", "Control", "Control", "LPS", "LPS", "LPS"))
)

# Define annotation colors for Genotype only
ann_colors <- list(
  Genotype = c("wt" = "#05d7df", "mut" = "#d6a3eb")
)

# Read the cell cycle gene data
cell_cycle_mef <- read.table('mef_cellcyclegenes.txt', header = TRUE)
cell_cycle_lmc <- read.table('lmc_cellcyclegenes.txt', header = TRUE)
colnames(cell_cycle_mef)[1] <- "Gene_name"
colnames(cell_cycle_lmc)[1] <- "Gene_name"

# Combine the gene data and convert gene names to uppercase
cell_cycle_mef_combo <- rbind(cell_cycle_lmc, cell_cycle_mef)
cell_cycle_mef_combo$Gene_name <- toupper(cell_cycle_mef_combo$Gene_name)

# Ensure rownames of countData are uppercase
rownames(countData) <- toupper(rownames(countData))

# Filter the countData to include only genes in the cell cycle list
cell_cycle_res <- intersect(rownames(countData), cell_cycle_mef_combo$Gene_name)
filteredCountData_cellcycle <- countData[cell_cycle_res, ]

# Subset the manual_annotation and filteredCountData_cellcycle to exclude LPS samples
samples_to_keep <- manual_annotation$Condition != "LPS"
filteredCountData_cellcycle <- filteredCountData_cellcycle[, samples_to_keep]
manual_annotation_filtered <- manual_annotation[samples_to_keep, "Genotype", drop = FALSE]

# Set the row names of manual_annotation_filtered to match the column names of filteredCountData_cellcycle
rownames(manual_annotation_filtered) <- colnames(filteredCountData_cellcycle)

# Check for any NA or infinite values and remove them
filteredCountData_cellcycle <- filteredCountData_cellcycle[complete.cases(filteredCountData_cellcycle), ]

# Ensure the data is not empty after filtering
if (nrow(filteredCountData_cellcycle) == 0 || ncol(filteredCountData_cellcycle) == 0) {
  stop("Filtered data contains no rows or columns. Please check your data and filtering criteria.")
}

# Plot the heatmap for the filtered data using only Genotype annotations
pheatmap(filteredCountData_cellcycle, 
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         annotation_col = manual_annotation_filtered, 
         show_rownames = TRUE, 
         scale = 'row',
         annotation_colors = ann_colors)


# Automatically added: save last plot as SVG
ggsave('figure_068.svg', width = 8, height = 6)


In [ ]:
#troubleshoot if needed

nf_kb <- read.table('nf_kb_sites.txt')
colnames(nf_kb)[1] <- "Gene_name"
# Filter the countData to include only genes in the nf_kb list
rownames(countData) <- toupper(rownames(countData))
nf_kb_genes <- intersect(rownames(countData), nf_kb$Gene_name)
filteredCountData <- countData[nf_kb_genes, ]

# Rank genes based on their mean expression levels and select the top 100
meanExpressionLevels <- rowMeans(filteredCountData)
top100Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]
filteredCountDataTop100 <- filteredCountData[top100Genes, ]

# Apply further filtering if necessary (you might want to skip this if focusing on top 100 already)
# Example pattern shown in your request - adjust if using this additional filter step
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredCountDataTop100), perl = TRUE)
filteredCountDataTop100Filtered <- filteredCountDataTop100[filtered_rows, ]




In [ ]:
library(pheatmap)

# Create the manual annotation dataframe
manual_annotation <- data.frame(
  Genotype = factor(c("wt", "wt", "wt", "wt", "wt", "wt", "wt", "wt", "wt", "wt", "mut", "mut", "mut", "mut", "mut", "mut")),
  Condition = factor(c("Control", "Control", "Control", "Control", "Control", "LPS", "LPS", "LPS", "LPS", "LPS", "Control", "Control", "Control", "LPS", "LPS", "LPS"))
)

# Define annotation colors
ann_colors <- list(
  Genotype = c("wt" = "#fe9388", "mut" = "#9bca1e"),
  Condition = c("Control" = "#05d8e1", "LPS" = "#e298fe")
)

rownames(manual_annotation) <- colnames(filteredCountDataTop100Filtered)
#plot the heatmap
pheatmap(filteredCountDataTop100Filtered,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         annotation_col = manual_annotation,
         annotation_colors = ann_colors,
         show_rownames = TRUE,
         scale = "row")
# Automatically added: save last plot as SVG
ggsave('figure_072.svg', width = 8, height = 6)


In [ ]:
res <- results(dds, name="genotypemut.conditionLPS")

In [ ]:
# Correctly filtering genes, accounting for NAs in 'padj'
filtered_genes <- res[!is.na(res$log2FoldChange) & !is.na(res$padj) & res$padj < 0.05, ]


In [ ]:
#The next part of code will help obtain the following answer.

I.The effect of treatment in wild-type.[This is for WT, treated compared with untreated.]

1.Genes upregulated with LPS in wild-type samples

2.Genes downregulated with LPS in wild-type samples

In [ ]:
res = results(dds, contrast=c("condition", "LPS", "Control"))
ix = which.min(res$padj) # most significant
res <- res[order(res$padj),] # sort
kable(res[1:5,-(3:4)])

In [ ]:
p <- barplot(assay(dds)[ix,],las=2, main=rownames(dds)[ ix  ]  )
#the plot below shows which gene is most prominent and how much it is expressed in each sample.
# Automatically added: save last plot as SVG
ggsave('figure_078.svg', width = 8, height = 6)


In [ ]:
res2 <- res
#reset par
par(mfrow=c(1,1))
# Make a basic volcano plot
with(res2, plot(log2FoldChange, -log10(padj), pch=20, main="Volcano plot", xlim=c(-5,10)))

# Add colored points: blue if padj<0.05, red if log2FC>1 and padj<0.05)
with(subset(res2, padj<.05 ), points(log2FoldChange, -log10(padj), pch=20, col="blue"))
with(subset(res2, padj<.05 & abs(log2FoldChange)>1), points(log2FoldChange, -log10(padj), pch=20, col="dark green"))
# Automatically added: save last plot as SVG
ggsave('figure_079.svg', width = 8, height = 6)


In [ ]:
#omit null values and only consider entries where adjusted p-value < 0.05.
res2 <- na.omit(res2)
mef_I <- res2
res2_I <- res2[res2$padj < 0.05, ]
res2_I_all <- res2

In [ ]:
# Number of genes to label
N <- 5  

# Sorting the data frame based on the criteria (e.g., padj)
sorted_indices <- order(res2$padj)

# Initializing labels to NA
res2$labels <- NA

# Select top N genes based on sorted indices
top_N_indices <- sorted_indices[1:N]
filtered_indices <- top_N_indices[grepl("^Gm|-ik$|Mir|^Rps", res2$SYMBOL[top_N_indices])]
num_more <- length(filtered_indices)
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|-ik$|Mir|^Rps", res2$SYMBOL[next_valid_indices])]

# Combining the valid top indices and the next valid indices to get top N labels
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])

# Filtering out NA indices
final_top_N_indices <- final_top_N_indices[!is.na(final_top_N_indices)]
final_top_N_indices <- final_top_N_indices[!is.na(res2$SYMBOL[final_top_N_indices])]

# Checking if any of the final_top_N_indices are NA or if corresponding SYMBOLs are NA
if (any(is.na(final_top_N_indices)) || any(is.na(res2$SYMBOL[final_top_N_indices]))) {
    stop("NA indices detected. Please resolve before proceeding.")
}

# Setting the labels for these final top N indices
res2$labels[final_top_N_indices] <- res2$SYMBOL[final_top_N_indices]

# Creating the EnhancedVolcano plot
EnhancedVolcano(res2,
                lab = rownames(res2),
                x = 'log2FoldChange',
                y = 'padj')

# Automatically added: save last plot as SVG
ggsave('figure_081.svg', width = 8, height = 6)


Here above plot suggests upregulated genes(positive Foldchange values) and downregulated genes(negative Foldchange values) in wildtype samples when subjected to LPS treatment.

In [ ]:
#Finding DE genes with adjusted p-value < 0.05 and LFC threshold = 0.20 and saving as csv files.

In [ ]:
res3 <- res2[complete.cases(res2),]
res3$abs_LFC <- abs(res3$log2FoldChange)
res4 <- res3[res3$abs_LFC > 1, ]
res5 <- res4[res4$padj < 0.05, ]  

res2_I_up <- res2[res2$log2FoldChange > 1 & res2$padj < 0.05, ]
res2_I_down <- res2[res2$log2FoldChange < -1 & res2$padj < 0.05, ]

In [ ]:
res2_I$abs_LFC <- abs(res2_I$log2FoldChange)
res2_I_sig <- res2_I[res2_I$abs_LFC > 1, ]
res2_I_sig

In [ ]:
write.csv(res2_I,'results_DEgenes_MEF_I_2020counts_v1.csv')
write.csv(rownames(res2_I_up),'upregulated_genes_MEF_I.csv')
write.csv(rownames(res2_I_down),'downregulated_genes_MEF_I.csv')

write.csv(res2_I_up,'upregulated_genes_MEF_I_rf.csv')
write.csv(res2_I_down,'downregulated_genes_MEF_I_rf.csv')

In [ ]:
# Number of genes to label
N <- 5  

# Sorting the data frame based on the criteria (e.g., padj)
sorted_indices <- order(res2_I$padj)

# Initializing labels to NA if the 'labels' column doesn't exist
if (!"labels" %in% colnames(res2_I)) {
  res2_I$labels <- NA  # Fixed typo here; changed res5 to res2_I
}

# Selecting top N genes based on sorted indices
top_N_indices <- sorted_indices[1:N]

# Identifying those among the top N that need to be filtered out
filtered_indices <- top_N_indices[grepl("^Gm|-ik$|Mir|^Rpl", res2_I$SYMBOL[top_N_indices])]
num_more <- length(filtered_indices)
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|-ik$|Mir|^Rpl", res2_I$SYMBOL[next_valid_indices])]
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])

# Checking if 'labels' column exists in res2_I before assigning labels
if ("labels" %in% colnames(res2_I)) {
  # Set the labels for these final top N indices, avoiding NA indices
  non_na_indices <- final_top_N_indices[!is.na(res2_I$SYMBOL[final_top_N_indices])]
  res2_I$labels[non_na_indices] <- res2_I$SYMBOL[non_na_indices]
} else {
  cat("The 'labels' column does not exist in res2_I.\n")
}

# Creating the EnhancedVolcano plot using res2_I and its labels
EnhancedVolcano(res2_I,
                lab = rownames(res2_I),
                x = 'log2FoldChange',
                y = 'padj',xlim = c(-12,12),ylim = c(0,350))

# Automatically added: save last plot as SVG
ggsave('figure_087.svg', width = 8, height = 6)


II.The effect of treatment in mutant samples [This is for Sp3 knockout,treated compared with untreated ]   


3.Genes upregulated with LPS in Sp3 knockout samples

4.Genes downregulated with LPS in Sp3 knockout samples


In [ ]:
res <- results(dds,list(c("condition_LPS_vs_Control","genotypemut.conditionLPS") ))
ix = which.min(res$padj) # most significant
res <- res[order(res$padj),] # sort
kable(res[1:5,-(3:4)])

In [ ]:
barplot(assay(dds)[ix,],las=2, main=rownames(dds)[ ix  ]  )
# Automatically added: save last plot as SVG
ggsave('figure_090.svg', width = 8, height = 6)


In [ ]:
res2 <- res
res2 <- na.omit(res2)
mef_II <- res2

In [ ]:
#filtering entries where adjusted p-value < 0.05 

res2_II <- res2[res2$padj < 0.05, ]
res2_II_all <- res2

In [ ]:
#reset par
par(mfrow=c(1,1))
with(res2, plot(log2FoldChange, -log10(padj), pch=20, main="Volcano plot", xlim=c(-5,10)))
# Add colored points: blue if padj<0.05, red if log2FC>1 and padj<0.05)
with(subset(res2, padj<.05 ), points(log2FoldChange, -log10(padj), pch=20, col="blue"))
with(subset(res2, padj<.05 & abs(log2FoldChange)>1), points(log2FoldChange, -log10(padj), pch=20, col="dark green"))
# Automatically added: save last plot as SVG
ggsave('figure_093.svg', width = 8, height = 6)


In [ ]:
# Number of genes to label
N <- 10  

# Sorting the data frame based on the criteria (e.g., padj)
sorted_indices <- order(res2$padj)

# Initializing labels to NA
res2$labels <- NA

# Selecting top N genes based on sorted indices
top_N_indices <- sorted_indices[1:N]

# Identifying those among the top N that need to be filtered out
filtered_indices <- top_N_indices[grepl("^Gm|-ik$|Mir", res2$SYMBOL[top_N_indices])]
num_more <- length(filtered_indices)
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|-ik$|Mir", res2$SYMBOL[next_valid_indices])]
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])
final_top_N_indices <- final_top_N_indices[!is.na(final_top_N_indices)]
final_top_N_indices <- final_top_N_indices[!is.na(res2$SYMBOL[final_top_N_indices])]

# Checking if any of the final_top_N_indices are NA or if corresponding SYMBOLs are NA
if (any(is.na(final_top_N_indices)) || any(is.na(res2$SYMBOL[final_top_N_indices]))) {
    stop("NA indices detected. Please resolve before proceeding.")
}

# Setting the labels for these final top N indices
res2$labels[final_top_N_indices] <- res2$SYMBOL[final_top_N_indices]

# Creating the EnhancedVolcano plot
EnhancedVolcano(res2,
                lab = rownames(res2),
                x = 'log2FoldChange',
                y = 'padj')

# Automatically added: save last plot as SVG
ggsave('figure_094.svg', width = 8, height = 6)


Here above plot suggests upregulated genes(positive Foldchange) and downregulated genes(negative Foldchange) in mutant samples when subjected to LPS treatment

In [ ]:
res3 <- res2[complete.cases(res2),]
res2_I_full <- res3
res3$abs_LFC <- abs(res3$log2FoldChange)
res4 <- res3[res3$abs_LFC > 0.20, ]
res5 <- res4[res4$padj < 1, ]  

res2_II_up <- res2[res2$log2FoldChange > 1 & res2$padj < 0.05, ]
res2_II_down <- res2[res2$log2FoldChange < -1 & res2$padj < 0.05, ]

write.csv(res2_II,'results_DEgenes_MEF_II_2020counts_v1.csv')
write.csv(rownames(res2_II_up),'upregulated_genes_MEF_II.csv')
write.csv(rownames(res2_II_down),'downregulated_genes_MEF_II.csv')


write.csv(res2_II_up,'upregulated_genes_MEF_II_rf.csv')
write.csv(res2_II_down,'downregulated_genes_MEF_II_rf.csv')


In [ ]:
res2_II$abs_LFC <- abs(res2_II$log2FoldChange)
res2_II_sig <- res2_II[res2_II$abs_LFC > 1, ]
res2_II_sig

In [ ]:
# Number of genes to label
N <- 10  

# Sorting the data frame based on the criteria (e.g., padj)
sorted_indices <- order(res2_II$padj)

# Initializing labels to NA if the 'labels' column doesn't exist
if (!"labels" %in% colnames(res2_II)) {
  res2_II$labels <- NA  # Changed res2_I to res2_II
}

# Selecting top N genes based on sorted indices
top_N_indices <- sorted_indices[1:N]

# Identifying those among the top N that need to be filtered out
filtered_indices <- top_N_indices[grepl("^Gm|Rik$|Mir", res2_II$SYMBOL[top_N_indices])]  # Changed res2_I to res2_II

num_more <- length(filtered_indices)
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|Rik$|Mir", res2_II$SYMBOL[next_valid_indices])]  # Changed res2_I to res2_II
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])

# Checking if 'labels' column exists in res2_II before assigning labels
if ("labels" %in% colnames(res2_II)) {  # Changed res2_I to res2_II
  # Set the labels for these final top N indices, avoiding NA indices
  non_na_indices <- final_top_N_indices[!is.na(res2_II$SYMBOL[final_top_N_indices])]  # Changed res2_I to res2_II
  res2_II$labels[non_na_indices] <- res2_II$SYMBOL[non_na_indices]  # Changed res2_I to res2_II
} else {
  cat("The 'labels' column does not exist in res2_II.\n")  # Changed res2_I to res2_II
}

# Creating the EnhancedVolcano plot using res2_II and its labels
EnhancedVolcano(res2_II,  # Changed res2_I to res2_II
                lab = rownames(res2_II),  # Changed res2_I to res2_II
                x = 'log2FoldChange',
                y = 'padj',ylim = c(0,350),xlim = c(-10,10))

# Automatically added: save last plot as SVG
ggsave('figure_098.svg', width = 8, height = 6)


In [ ]:
lmc_data <- read.csv('merged_LMC_I_II_III_deseq2results.csv')

In [ ]:
lmc_data$X <- NULL

In [ ]:
res = results(dds, contrast=c("genotype","mut","wt"))
ix = which.min(res$padj) # most significant
res <- res[order(res$padj),] # sort
kable(res[1:5,-(3:4)])

In [ ]:
barplot(assay(dds)[ix,],las=2, main=rownames(dds)[ ix  ]  )
# Automatically added: save last plot as SVG
ggsave('figure_103.svg', width = 8, height = 6)


In [ ]:
res2 <- res
res2 <- na.omit(res2)
mef_III <- res2

In [ ]:
res2_III <- res2[res2$padj < 0.05, ]
res2_III_all <- res2

In [ ]:
#reset par
par(mfrow=c(1,1))
with(res2, plot(log2FoldChange, -log10(padj), pch=20, main="Volcano plot", xlim=c(-5,10)))

# Add colored points: blue if padj<0.05, red if log2FC>1 and padj<0.05)
with(subset(res2, padj<.05 ), points(log2FoldChange, -log10(padj), pch=20, col="blue"))
with(subset(res2, padj<.05 & abs(log2FoldChange)>1), points(log2FoldChange, -log10(padj), pch=20, col="dark green"))
# Automatically added: save last plot as SVG
ggsave('figure_106.svg', width = 8, height = 6)


In [ ]:
countData

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(stringr)
})

###########################
## 0. Standardize countData
###########################
# Force to data.frame (drop tibble weirdness)
countData_df <- as.data.frame(countData, check.names = FALSE)

# If there is no rownames OR rownames are just 1:n,
# but there is a first column that looks like gene names,
# promote that first column to rownames.
if ( (is.null(rownames(countData_df)) ||
      all(rownames(countData_df) == as.character(seq_len(nrow(countData_df)))) )
     &&
     # heuristic: first column looks like gene symbols (A-Z,0-9,-,.)
     any(grepl("^[A-Za-z0-9._-]+$", countData_df[[1]]))
) {
  # make first column rownames
  rownames(countData_df) <- countData_df[[1]]
  countData_df <- countData_df[ , -1, drop = FALSE]
}

# Now rownames should be gene names.
# Uppercase them so they match HOX/POU list.
rownames(countData_df) <- toupper(rownames(countData_df))

###########################
## 1. Keep only numeric columns
###########################
numeric_cols_idx <- sapply(countData_df, function(x) is.numeric(x) || is.integer(x))

# Subset using that index, BUT first make sure lengths align
if (length(numeric_cols_idx) != ncol(countData_df)) {
  stop("Column/type mismatch after coercion. Something is structurally off in countData.")
}

countData_num <- countData_df[, numeric_cols_idx, drop = FALSE]

cat("Step 1: numeric columns kept =", ncol(countData_num), "of", ncol(countData_df), "\n")

if (ncol(countData_num) == 0) {
  stop("No numeric expression columns found after cleaning. Cannot proceed.")
}

###########################
## 2. Define genes of interest
###########################
genes_of_interest <- c(
  "HOXA1","HOXA10","HOXA11","HOXA11OS","HOXA2","HOXA3","HOXA4","HOXA5","HOXA6","HOXA7","HOXA9",
  "HOXB1","HOXB2","HOXB3","HOXB3OS","HOXB4","HOXB5","HOXB5OS","HOXB6","HOXB7","HOXB8","HOXB9",
  "HOXC10","HOXC11","HOXC13","HOXC4","HOXC5","HOXC6","HOXC8","HOXC9",
  "HOXD10","HOXD11","HOXD3","HOXD3OS1","HOXD8","HOXD9",
  "PAX3","PAX6",
  "POU2F1","POU2F2","POU2F3",
  "POU3F1","POU3F2","POU3F3","POU3F4",
  "POU4F1","POU4F3",
  "POU5F2",
  "POU6F1"
)

row_idx <- match(genes_of_interest, rownames(countData_num))
row_idx <- row_idx[!is.na(row_idx)]
expr_sub <- countData_num[row_idx, , drop = FALSE]

cat("Step 2: genes found =", nrow(expr_sub), "out of", length(genes_of_interest), "\n")
if (nrow(expr_sub) == 0) {
  stop("None of your HOX/PAX/POU genes are present after cleaning.")
}

###########################
## 3. Make sure matrix is numeric
###########################
expr_sub_num <- as.data.frame(
  lapply(expr_sub, function(col) as.numeric(as.character(col))),
  check.names = FALSE
)
expr_sub_num <- as.matrix(expr_sub_num)
rownames(expr_sub_num) <- rownames(expr_sub)
colnames(expr_sub_num) <- colnames(expr_sub)

###########################
## 4. Infer biological grouping from column names
##    Genotype: WT / MUT / etc.
##    Condition: Control / LPS
###########################
sample_names <- colnames(expr_sub_num)

geno_vec <- ifelse(grepl("WT",  sample_names, ignore.case = TRUE), "WT",
            ifelse(grepl("MUT", sample_names, ignore.case = TRUE), "MUT", "UNK"))

cond_vec <- ifelse(grepl("LPS", sample_names, ignore.case = TRUE), "LPS",
            ifelse(grepl("CTRL|CONTROL", sample_names, ignore.case = TRUE), "Control", "UNK"))

group_key <- paste(geno_vec, cond_vec, sep = "_")

cat("Step 3: unique groups detected from column names:\n")
print(unique(group_key))

###########################
## 5. Collapse replicates by group_key
###########################
collapse_by_key <- function(mat, keys) {
  split_cols <- split(seq_along(keys), keys)
  collapsed <- sapply(split_cols, function(idx) {
    rowMeans(mat[, idx, drop = FALSE], na.rm = TRUE)
  })
  as.matrix(collapsed)
}

expr_collapsed <- collapse_by_key(expr_sub_num, group_key)

# force a nice canonical order if present
nice_order <- c("WT_Control","WT_LPS","MUT_Control","MUT_LPS","UNK_UNK")
nice_order <- nice_order[nice_order %in% colnames(expr_collapsed)]
expr_collapsed <- expr_collapsed[, nice_order, drop = FALSE]

cat("Step 4: collapsed matrix dims (genes x conditions): ", dim(expr_collapsed), "\n")

###########################
## 6. Build annotation for columns (top bars)
###########################
split_df <- do.call(rbind, strsplit(colnames(expr_collapsed), "_"))
colnames(split_df) <- c("Genotype","Condition")
annotation_col <- as.data.frame(split_df, stringsAsFactors = FALSE)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 7. Annotation colors
###########################
ann_colors <- list(
  Genotype = c(
    WT  = "#66c2a5",
    MUT = "#fc8d62",
    UNK = "grey80"
  ),
  Condition = c(
    Control = "#8da0cb",
    LPS     = "#e78ac3",
    UNK     = "grey90"
  )
)

###########################
## 8. Heatmap palette
###########################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

###########################
## 9. Plot heatmap
###########################
pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "row",   # per-gene z-score across conditions
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 22,
  border_color = NA,
  legend = TRUE,
  main = "HOX / PAX / POU Gene Expression (from countData)",
  treeheight_row = 10,
  treeheight_col = 12
)

###########################
## 10. Inspect raw averaged expression
###########################
cat("\nRaw averaged expression (not z-scored), first few genes:\n")
print(round(expr_collapsed[1:min(5,nrow(expr_collapsed)), , drop = FALSE], 2))

# Automatically added: save last plot as SVG
ggsave('figure_108.svg', width = 8, height = 6)


In [ ]:
# Number of genes to label
N <- 10  

# Sorting the data frame based on the criteria (e.g., padj)
sorted_indices <- order(res2$padj)

# Initializing labels to NA
res2$labels <- NA

# Selecting top N genes based on sorted indices
top_N_indices <- sorted_indices[1:N]

# Identifying those among the top N that need to be filtered out
filtered_indices <- top_N_indices[grepl("^Gm|Rik$|Mir|^Rpl", res2$SYMBOL[top_N_indices])]

# Identifying how many more genes we need to pick
num_more <- length(filtered_indices)

# Picking the next top-most genes that are valid
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|Rik$|Mir|^Rpl", res2$SYMBOL[next_valid_indices])]

# Combining the valid top indices and the next valid indices to get top N labels
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])

# Filtering out NA indices
final_top_N_indices <- final_top_N_indices[!is.na(final_top_N_indices)]
final_top_N_indices <- final_top_N_indices[!is.na(res2$SYMBOL[final_top_N_indices])]

# Checking if any of the final_top_N_indices are NA or if corresponding SYMBOLs are NA
if (any(is.na(final_top_N_indices)) || any(is.na(res2$SYMBOL[final_top_N_indices]))) {
    stop("NA indices detected. Please resolve before proceeding.")
}

# Setting the labels for these final top N indices
res2$labels[final_top_N_indices] <- res2$SYMBOL[final_top_N_indices]

# Creating the EnhancedVolcano plot
EnhancedVolcano(res2,
                lab = rownames(res2),
                x = 'log2FoldChange',
                y = 'padj')

# Automatically added: save last plot as SVG
ggsave('figure_109.svg', width = 8, height = 6)


Here above results suggest upregulated genes(positive Foldchange) and downregulated genes(negative Foldchange)in
Wildtype samples compared to mutant without considering control/LPS treatment.


In [ ]:
res3 <- res2[complete.cases(res2),]
res2_II_full <- res3
res3$abs_LFC <- abs(res3$log2FoldChange)
res4 <- res3[res3$abs_LFC > 0.20, ]
res5 <- res4[res4$padj < 0.05, ]  


res3_up <- res2[res2$log2FoldChange > 1 & res2$padj < 0.05, ]
res3_down <- res2[res2$log2FoldChange < -1 & res2$padj < 0.05, ]

write.csv(res2_III,'MEF_III_DEgenes.csv')
write.csv(rownames(res3_up),'upregulated_genes_MEF_III.csv')
write.csv(rownames(res3_down),'downregulated_genes_MEF_III.csv')


write.csv(res3_up,'upregulated_genes_MEF_III_rf.csv')
write.csv(res3_down,'downregulated_genes_MEF_III_rf.csv')

res3_up_v <- res2[res2$log2FoldChange > 0 & res2$padj < 0.05, ]
res3_down_v <- res2[res2$log2FoldChange < 0 & res2$padj < 0.05, ]

write.csv(rownames(res3_up_v),'upregulated_genes_MEF_III_venndiagram.csv')
write.csv(rownames(res3_down_v),'downregulated_genes_MEF_III_venndiagram.csv')

In [ ]:
# Load required packages
library(plotly)
library(htmltools)

DE_results <- res2_III
# Ensure gene names are uppercase
rownames(countData2) <- toupper(rownames(countData2))
rownames(DE_results) <- toupper(rownames(DE_results))

# Filter to only genes present in countData1
gene_list <- rownames(DE_results)
gene_list <- gene_list[gene_list %in% rownames(countData2)]

# Subset countData1 accordingly
countData3 <- countData2[gene_list, , drop = FALSE]

# Extract sample metadata
sample_ids <- colnames(countData3)
Group <- groups
Genotype <- ifelse(grepl("wt", Group, ignore.case = TRUE), "WT", "MUT")
Condition <- ifelse(grepl("Control", Group, ignore.case = TRUE), "Control", "LPS")

In [ ]:
res2_III_hmap <- res2_III
res2_III_hmap$abs_LFC <- abs(res2_III$log2FoldChange)
res2_III_hmap <- res2_III_hmap[res2_III_hmap$abs_LFC > 2, ]
#res2_III_hmap <- res2_III_hmap[res2_III_hmap$abs_LFC > 3, ]

In [ ]:
columns_without_LPS <- !grepl("Lps", names(countData1))

# Select only those columns
countData1_onlycontrol <- countData1[, columns_without_LPS]

In [ ]:
MEF_DEgenes_III <- intersect(rownames(countData1_onlycontrol), rownames(res2_III_hmap))

In [ ]:
filtered_countdata <- countData1_onlycontrol[MEF_DEgenes_III, ]
nrow(filtered_countdata)
filtered_countdata_matrix <- as.matrix(filtered_countdata)

# Define the pattern to remove unwanted genes
pattern <- "^(Gm|MIR|ENSMUS|Rpl)|Rik$|^[A-Z]+$|\\."
rows_to_keep <- grep(pattern, rownames(filtered_countdata_matrix), invert = TRUE, ignore.case = TRUE)

# Further filter to keep only the first 500 after removing unwanted genes
filtered_matrix <- filtered_countdata_matrix[rows_to_keep, ]

# Save the filtered matrix to CSV
write.csv(filtered_matrix, "filter_MEF.csv")

# Generate heatmap and extract clustering information, reducing font size
heatmap_result <- pheatmap(filtered_matrix, 
                           scale = "row",
                           cluster_cols = FALSE, # Do not cluster columns
                           show_rownames = TRUE,
                           show_colnames = TRUE,
                           main = "Heatmap of DE genes in MEF (Wildtype versus Sp3 knockout)",
                           fontsize_row = 5,   # Adjust row label font size
                           fontsize_col = 8,   # Adjust column label font size
                           width = 20,         # Width of the plot in inches
                           height = 40)        # Height of the plot in inches

# Extract clustering order of rows
clustered_order <- heatmap_result$tree_row$order

# Reorder matrix according to clustering order
filtered_matrix_ordered <- filtered_matrix[clustered_order, ]

# Save reordered matrix to CSV
#write.csv(filtered_matrix_ordered, "filter_MEF_ordered.csv")



# Automatically added: save last plot as SVG
ggsave('figure_117.svg', width = 8, height = 6)


In [ ]:

# Convert row names to uppercase to match case insensitivity
rownames(countData) <- toupper(rownames(countData))
rownames(res2_II_sig) <- toupper(rownames(res2_II_sig))
nf_kb$Gene_name <- toupper(nf_kb$Gene_name)  # Ensuring case consistency

common_genes_sig_II <- intersect(rownames(countData), rownames(res2_II_sig))


# Also intersect with nf_kb gene names
common_genes_nf_kb <- intersect(rownames(countData), nf_kb$Gene_name)

# Get the intersection of the two lists to find genes common in both
common_genes_final <- intersect(common_genes_sig_II, common_genes_nf_kb)

# Filter countData to include only rows that have genes present in both res2_I_sig and nf_kb
filteredCountData <- countData[common_genes_final, ]

# Further filter rows to exclude certain gene name prefixes
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredCountData), perl = TRUE)
filteredCountDataFiltered <- filteredCountData[filtered_rows, ]

meanExpressionLevels <- rowMeans(filteredCountDataFiltered)

# Sort genes by mean expression levels in decreasing order and select the top 50
top50Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))[1:50]

# Filter the data to include only these top 50 genes
top50FilteredData <- filteredCountDataFiltered[top50Genes, ]

# Plot the heatmap of the top 50 filtered countData
pheatmap(top50FilteredData,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
       #  scale = "row",
         fontsize_row = 8)
# Automatically added: save last plot as SVG
ggsave('figure_118.svg', width = 8, height = 6)


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(stringr)
  library(gridExtra)
  library(grid)
})

###########################
## 0. Prepare countData as a clean numeric matrix
###########################
# Convert to data.frame so we can manipulate safely
countData_df <- as.data.frame(countData, check.names = FALSE)

# If rownames are missing or just 1,2,3..., but first column looks like gene names,
# promote that first column to rownames
if ((is.null(rownames(countData_df)) ||
     all(rownames(countData_df) == as.character(seq_len(nrow(countData_df))))) &&
    any(grepl("^[A-Za-z0-9._-]+$", countData_df[[1]]))
) {
  rownames(countData_df) <- countData_df[[1]]
  countData_df <- countData_df[, -1, drop = FALSE]
}

# Uppercase gene names to match gene list
rownames(countData_df) <- toupper(rownames(countData_df))

###########################
## 1. Keep only numeric columns (expression columns)
###########################
numeric_cols_idx <- sapply(countData_df, function(x) is.numeric(x) || is.integer(x))
countData_num <- countData_df[, numeric_cols_idx, drop = FALSE]

cat("Step 1: numeric columns kept =", ncol(countData_num), "of", ncol(countData_df), "\n")
if (ncol(countData_num) == 0) {
  stop("No numeric expression columns found after cleaning.")
}

###########################
## 2. Specify genes of interest (HOX / PAX / POU panel or other panel)
###########################
genes_of_interest <- c(
  "HOXA1","HOXA10","HOXA11","HOXA11OS","HOXA2","HOXA3","HOXA4","HOXA5","HOXA6","HOXA7","HOXA9",
  "HOXB1","HOXB2","HOXB3","HOXB3OS","HOXB4","HOXB5","HOXB5OS","HOXB6","HOXB7","HOXB8","HOXB9",
  "HOXC10","HOXC11","HOXC13","HOXC4","HOXC5","HOXC6","HOXC8","HOXC9",
  "HOXD10","HOXD11","HOXD3","HOXD3OS1","HOXD8","HOXD9",
  "PAX3","PAX6",
  "POU2F1","POU2F2","POU2F3",
  "POU3F1","POU3F2","POU3F3","POU3F4",
  "POU4F1","POU4F3",
  "POU5F2",
  "POU6F1"
)

# Match in the user-specified order
row_idx <- match(genes_of_interest, rownames(countData_num))
row_idx <- row_idx[!is.na(row_idx)]
expr_sub <- countData_num[row_idx, , drop = FALSE]

cat("Step 2: genes found =", nrow(expr_sub), "out of", length(genes_of_interest), "\n")
if (nrow(expr_sub) == 0) {
  stop("None of the requested genes were found in countData.")
}

###########################
## 3. Force numeric (defensive against factors)
###########################
expr_sub_num <- as.data.frame(
  lapply(expr_sub, function(col) as.numeric(as.character(col))),
  check.names = FALSE
)
expr_sub_num <- as.matrix(expr_sub_num)
rownames(expr_sub_num) <- rownames(expr_sub)
colnames(expr_sub_num) <- colnames(expr_sub)

###########################
## 4. Infer genotype and condition from column names
##    We assume your sample columns are named such that:
##       - "WT" or "wt" appears in WT samples
##       - "MUT" or "mut" appears in mutant samples
##       - "Control", "CTRL", etc. for control condition
##       - "LPS" for stimulated condition
###########################
sample_names <- colnames(expr_sub_num)

geno_vec <- ifelse(grepl("WT",  sample_names, ignore.case = TRUE), "wt",
            ifelse(grepl("MUT", sample_names, ignore.case = TRUE), "mut", "unk"))

cond_vec <- ifelse(grepl("LPS", sample_names, ignore.case = TRUE), "LPS",
            ifelse(grepl("CTRL|CONTROL", sample_names, ignore.case = TRUE), "Control", "unk"))

# Each sample is now assigned to a biological group like "wt_Control" or "mut_LPS"
group_key <- paste(geno_vec, cond_vec, sep = "_")

cat("Step 3: unique groups detected in column names:\n")
print(unique(group_key))

###########################
## 5. Collapse replicates by group (average columns that map to same group_key)
###########################
collapse_by_key <- function(mat, keys) {
  split_cols <- split(seq_along(keys), keys)
  collapsed <- sapply(split_cols, function(idx) {
    rowMeans(mat[, idx, drop = FALSE], na.rm = TRUE)
  })
  as.matrix(collapsed)
}

expr_collapsed <- collapse_by_key(expr_sub_num, group_key)

# Coerce colnames to character (safety for downstream)
mode(colnames(expr_collapsed)) <- "character"

cat("Step 4: collapsed matrix dims (genes x conditions):",
    paste(dim(expr_collapsed), collapse = " x "), "\n")
cat("Collapsed column names:\n")
print(colnames(expr_collapsed))

###########################
## 6. Build annotation data.frame from collapsed column names
##    e.g. "wt_Control" -> Genotype = wt, Condition = Control
###########################
split_df <- do.call(rbind, strsplit(colnames(expr_collapsed), "_"))
colnames(split_df) <- c("Genotype","Condition")
annotation_col <- as.data.frame(split_df, stringsAsFactors = FALSE)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 7. Define annotation colors
###########################
ann_colors <- list(
  Condition = c(
    Control = "#8da0cb",
    LPS     = "#e78ac3",
    unk     = "grey90"
  ),
  Genotype = c(
    wt  = "#b3de69",  # light green
    mut = "#bc80bd",  # mauve/purple
    unk = "grey80"
  )
)

###########################
## 8. Define heatmap color palettes
###########################

## Diverging palette for row-scaled version (blue -> white -> red)
hm_diverging <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

## Sequential palette for absolute version (blue -> red-yellow high)
hm_sequential <- colorRampPalette(
  c("#08306B", "#2879B9", "#72B2D7", "#FDD67B", "#F03B20")
)(200)

###########################
## 9. Generate two heatmaps:
##    A) absolute expression (no scaling)
##    B) row-scaled (z-score per gene)
###########################

# heatmap A (absolute expression, scale="none")
heat_abs <- pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "none",                       # raw magnitude
  color = hm_sequential,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 6.5,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 22,
  border_color = NA,
  legend = TRUE,
  main = "Absolute expression",
  treeheight_row = 10,
  treeheight_col = 12
)

# heatmap B (relative expression, scale="row")
heat_rel <- pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "row",                        # z-score within each gene
  color = hm_diverging,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 6.5,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 22,
  border_color = NA,
  legend = TRUE,
  main = "Row-scaled (per-gene z-score)",
  treeheight_row = 10,
  treeheight_col = 12
)

###########################
## 10. Arrange side by side into one figure
###########################
# pheatmap() returns a list; the gtable is at [[4]]
g_abs <- heat_abs[[4]]
g_rel <- heat_rel[[4]]

grid.newpage()
grid.arrange(g_abs, g_rel, ncol = 2)

###########################
## 11. (Optional) Save side-by-side to file
## Use png() for notebook / Colab compatibility
###########################

png("HOX_PAX_POU_dual_heatmap.png", width = 2400, height = 1200, res = 300)
grid.arrange(g_abs, g_rel, ncol = 2)

cat("Saved side-by-side comparison as 'HOX_PAX_POU_dual_heatmap.png'\n")

# Automatically added: save last plot as SVG
ggsave('figure_119.svg', width = 8, height = 6)


In [ ]:
colnames(countData_df)

In [ ]:
metadata

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(stringr)
  library(gridExtra)
  library(grid)
})

###########################
## 0. Prepare countData as a clean numeric matrix
###########################
# Convert to data.frame so we can manipulate safely
countData_df <- as.data.frame(countData, check.names = FALSE)

# If rownames are missing or just 1,2,3..., but first column looks like gene names,
# promote that first column to rownames
if ((is.null(rownames(countData_df)) ||
     all(rownames(countData_df) == as.character(seq_len(nrow(countData_df))))) &&
    any(grepl("^[A-Za-z0-9._-]+$", countData_df[[1]]))
) {
  rownames(countData_df) <- countData_df[[1]]
  countData_df <- countData_df[, -1, drop = FALSE]
}

# Uppercase gene names to match gene list
rownames(countData_df) <- toupper(rownames(countData_df))

###########################
## 1. Keep only numeric columns (expression columns)
###########################
numeric_cols_idx <- sapply(countData_df, function(x) is.numeric(x) || is.integer(x))
countData_num <- countData_df[, numeric_cols_idx, drop = FALSE]

cat("Step 1: numeric columns kept =", ncol(countData_num), "of", ncol(countData_df), "\n")
if (ncol(countData_num) == 0) {
  stop("No numeric expression columns found after cleaning.")
}

###########################
## 2. Specify genes of interest (HOX / PAX / POU panel or other panel)
###########################
genes_of_interest <- c(
  "HOXA1","HOXA10","HOXA11","HOXA11OS","HOXA2","HOXA3","HOXA4","HOXA5","HOXA6","HOXA7","HOXA9",
  "HOXB1","HOXB2","HOXB3","HOXB3OS","HOXB4","HOXB5","HOXB5OS","HOXB6","HOXB7","HOXB8","HOXB9",
  "HOXC10","HOXC11","HOXC13","HOXC4","HOXC5","HOXC6","HOXC8","HOXC9",
  "HOXD10","HOXD11","HOXD3","HOXD3OS1","HOXD8","HOXD9",
  "PAX3","PAX6",
  "POU2F1","POU2F2","POU2F3",
  "POU3F1","POU3F2","POU3F3","POU3F4",
  "POU4F1","POU4F3",
  "POU5F2",
  "POU6F1"
)

# Match in the user-specified order
row_idx <- match(genes_of_interest, rownames(countData_num))
row_idx <- row_idx[!is.na(row_idx)]
expr_sub <- countData_num[row_idx, , drop = FALSE]

cat("Step 2: genes found =", nrow(expr_sub), "out of", length(genes_of_interest), "\n")
if (nrow(expr_sub) == 0) {
  stop("None of the requested genes were found in countData.")
}

###########################
## 3. Force numeric (defensive against factors)
###########################
expr_sub_num <- as.data.frame(
  lapply(expr_sub, function(col) as.numeric(as.character(col))),
  check.names = FALSE
)
expr_sub_num <- as.matrix(expr_sub_num)
rownames(expr_sub_num) <- rownames(expr_sub)
colnames(expr_sub_num) <- colnames(expr_sub)

###########################
## 4. Infer genotype and condition from column names
##    We assume your sample columns are named such that:
##       - "WT" or "wt" appears in WT samples
##       - "MUT" or "mut" appears in mutant samples
##       - "Control", "CTRL", etc. for control condition
##       - "LPS" for stimulated condition
###########################
sample_names <- colnames(expr_sub_num)

geno_vec <- ifelse(grepl("WT",  sample_names, ignore.case = TRUE), "wt",
            ifelse(grepl("MUT", sample_names, ignore.case = TRUE), "mut", "unk"))

cond_vec <- ifelse(grepl("LPS", sample_names, ignore.case = TRUE), "LPS",
            ifelse(grepl("CTRL|CONTROL", sample_names, ignore.case = TRUE), "Control", "unk"))

# Each sample is now assigned to a biological group like "wt_Control" or "mut_LPS"
group_key <- paste(geno_vec, cond_vec, sep = "_")

cat("Step 3: unique groups detected in column names:\n")
print(unique(group_key))

###########################
## 5. Collapse replicates by group (average columns that map to same group_key)
###########################
collapse_by_key <- function(mat, keys) {
  split_cols <- split(seq_along(keys), keys)
  collapsed <- sapply(split_cols, function(idx) {
    rowMeans(mat[, idx, drop = FALSE], na.rm = TRUE)
  })
  as.matrix(collapsed)
}

expr_collapsed <- collapse_by_key(expr_sub_num, group_key)

# Coerce colnames to character (safety for downstream)
mode(colnames(expr_collapsed)) <- "character"

cat("Step 4: collapsed matrix dims (genes x conditions):",
    paste(dim(expr_collapsed), collapse = " x "), "\n")
cat("Collapsed column names:\n")
print(colnames(expr_collapsed))

###########################
## 6. Build annotation data.frame from collapsed column names
##    e.g. "wt_Control" -> Genotype = wt, Condition = Control
###########################
split_df <- do.call(rbind, strsplit(colnames(expr_collapsed), "_"))
colnames(split_df) <- c("Genotype","Condition")
annotation_col <- as.data.frame(split_df, stringsAsFactors = FALSE)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 7. Define annotation colors
###########################
ann_colors <- list(
  Condition = c(
    Control = "#8da0cb",
    LPS     = "#e78ac3",
    unk     = "grey90"
  ),
  Genotype = c(
    wt  = "#b3de69",  # light green
    mut = "#bc80bd",  # mauve/purple
    unk = "grey80"
  )
)

###########################
## 8. Define heatmap color palettes
###########################

## Diverging palette for row-scaled version (blue -> white -> red)
hm_diverging <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

## Sequential palette for absolute version (blue -> red-yellow high)
hm_sequential <- colorRampPalette(
  c("#08306B", "#2879B9", "#72B2D7", "#FDD67B", "#F03B20")
)(200)

###########################
## 9. Generate two heatmaps:
##    A) absolute expression (no scaling)
##    B) row-scaled (z-score per gene)
###########################

# heatmap A (absolute expression, scale="none")
heat_abs <- pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "none",                       # raw magnitude
  color = hm_sequential,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 6.5,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 22,
  border_color = NA,
  legend = TRUE,
  main = "Absolute expression",
  treeheight_row = 10,
  treeheight_col = 12
)

# heatmap B (relative expression, scale="row")
heat_rel <- pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "row",                        # z-score within each gene
  color = hm_diverging,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 6.5,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 22,
  border_color = NA,
  legend = TRUE,
  main = "Row-scaled (per-gene z-score)",
  treeheight_row = 10,
  treeheight_col = 12
)

###########################
## 10. Arrange side by side into one figure
###########################
# pheatmap() returns a list; the gtable is at [[4]]
g_abs <- heat_abs[[4]]
g_rel <- heat_rel[[4]]

grid.newpage()
grid.arrange(g_abs, g_rel, ncol = 2)

###########################
## 11. (Optional) Save side-by-side to file
## Use png() for notebook / Colab compatibility
###########################

png("HOX_PAX_POU_dual_heatmap.png", width = 2400, height = 1200, res = 300)
grid.arrange(g_abs, g_rel, ncol = 2)

cat("Saved side-by-side comparison as 'HOX_PAX_POU_dual_heatmap.png'\n")

# Automatically added: save last plot as SVG
ggsave('figure_120.svg', width = 8, height = 6)


In [ ]:
#!/usr/bin/env Rscript

suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(stringr)
  library(grid)
})

stopifnot(exists("countData"))
stopifnot(exists("metadata"))

out_svg <- "HOX_PAX_POU_heatmap_mef.svg"

# -----------------------------
# HOX / PAX / POU genes ONLY
# -----------------------------
genes_of_interest <- c(
  "HOXA1","HOXA10","HOXA11","HOXA11OS","HOXA2","HOXA3","HOXA4","HOXA5","HOXA6","HOXA7","HOXA9",
  "HOXB1","HOXB2","HOXB3","HOXB3OS","HOXB4","HOXB5","HOXB5OS","HOXB6","HOXB7","HOXB8","HOXB9",
  "HOXC10","HOXC11","HOXC13","HOXC4","HOXC5","HOXC6","HOXC8","HOXC9",
  "HOXD10","HOXD11","HOXD3","HOXD3OS1","HOXD8","HOXD9",
  "PAX3","PAX6",
  "POU2F1","POU2F2","POU2F3",
  "POU3F1","POU3F2","POU3F3","POU3F4",
  "POU4F1","POU4F3",
  "POU5F2",
  "POU6F1"
)

# -----------------------------
# Clean matrix
# -----------------------------
countData_df <- as.data.frame(countData, check.names = FALSE)
rownames(countData_df) <- toupper(rownames(countData_df))

expr_mat <- as.matrix(countData_df)
mode(expr_mat) <- "numeric"

# -----------------------------
# Subset genes
# -----------------------------
genes_of_interest <- toupper(genes_of_interest)
expr_sub <- expr_mat[intersect(genes_of_interest, rownames(expr_mat)), , drop = FALSE]

# -----------------------------
# Collapse using metadata$id
# -----------------------------
metadata2 <- metadata %>%
  mutate(
    id = str_trim(as.character(id)),
    genotype = tolower(genotype),
    condition = tolower(condition),
    group = paste(genotype, condition, sep = "_")
  )

groups <- c("wt_control", "wt_lps", "mut_control", "mut_lps")

expr4 <- sapply(groups, function(g) {
  samps <- metadata2$id[metadata2$group == g]
  rowMeans(expr_sub[, samps, drop = FALSE], na.rm = TRUE)
}) |> as.matrix()

colnames(expr4) <- c("WT\n−", "WT\n+", "Sp3−/−\n−", "Sp3−/−\n+")

# -----------------------------
# Row Z-score
# -----------------------------
z <- function(x){
  s <- sd(x)
  if (is.na(s) || s == 0) return(rep(0,length(x)))
  (x - mean(x)) / s
}
expr4_z <- t(apply(expr4, 1, z))

# cap at ±1.5 to match visual intensity of paper
cap <- 1.5
expr4_z <- pmax(pmin(expr4_z, cap), -cap)

# -----------------------------
# EXACT paper-like color palette
# -----------------------------
paper_cols <- colorRampPalette(c(
  "#08306B",  # deep navy
  "#4292C6",  # blue
  "#F7F7F7",  # white center
  "#FDAE6B",  # orange
  "#A50F15"   # deep red
))(200)

breaks <- seq(-cap, cap, length.out = 201)

# -----------------------------
# SAVE TRUE SVG
# -----------------------------
svg(out_svg, width = 4.8, height = max(4.5, 0.15*nrow(expr4_z)+1.5))

pheatmap(
  expr4_z,
  color = paper_cols,
  breaks = breaks,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  gaps_col = 2,                 # WT | Sp3−/− separation
  border_color = NA,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 11,
  legend_breaks = c(-1,0,1),
  legend_labels = c("-1","0","1"),
  main = "HOX / PAX / POU genes"
)

grid.text("LPS:", x = unit(0.08,"npc"), y = unit(0.965,"npc"),
          gp = gpar(fontsize = 12))

dev.off()

cat("Saved SVG →", normalizePath(out_svg), "\n")


In [ ]:
library(pheatmap)
library(RColorBrewer)
library(grid)

# ---- color palette EXACTLY like the paper ----
paper_cols <- colorRampPalette(
  rev(brewer.pal(11, "RdBu"))
)(100)

# z-score limits used in the paper (important)
lim <- 1
bk <- seq(-lim, lim, length.out = 101)

svg("HOX_PAX_POU_heatmap.svg",
    width = 4.8,
    height = max(4.5, 0.15 * nrow(expr4_z) + 1.5))

pheatmap(
  expr4_z,
  color = paper_cols,
  breaks = bk,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  gaps_col = 2,
  border_color = NA,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 11,
  legend_breaks = c(-1, 0, 1),
  legend_labels = c("-1", "0", "1"),
  main = "HOX / PAX / POU genes"
)

grid.text("LPS:", x = unit(0.08, "npc"),
          y = unit(0.965, "npc"),
          gp = gpar(fontsize = 12))

dev.off()


In [ ]:
library(pheatmap)
library(grid)

svg("HOX_PAX_POU_heatmap.svg",
    width = 4.8,
    height = max(4.5, 0.15 * nrow(expr4_z) + 1.5))

pheatmap(
  expr4_z,
  color = paper_cols,
  breaks = bk,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  gaps_col = 2,
  border_color = NA,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 11,
  legend_breaks = c(-1, 0, 1),
  legend_labels = c("-1", "0", "1"),
  main = "HOX / PAX / POU genes"
)

grid.text("LPS:", x = unit(0.08, "npc"),
          y = unit(0.965, "npc"),
          gp = gpar(fontsize = 12))

dev.off()


In [ ]:
metadata$id

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(dplyr)
  library(stringr)
  library(gridExtra)
  library(grid)
})

###########################
## 0. Assume previous objects exist:
##    - countData
##    - expr_collapsed does NOT exist yet in this clean version
## We'll rebuild everything cleanly as before,
## but simplify for no-genotype case.
###########################

## --- same cleaning steps as before ---

countData_df <- as.data.frame(countData, check.names = FALSE)

if ((is.null(rownames(countData_df)) ||
     all(rownames(countData_df) == as.character(seq_len(nrow(countData_df))))) &&
    any(grepl("^[A-Za-z0-9._-]+$", countData_df[[1]]))
) {
  rownames(countData_df) <- countData_df[[1]]
  countData_df <- countData_df[, -1, drop = FALSE]
}

rownames(countData_df) <- toupper(rownames(countData_df))

numeric_cols_idx <- sapply(countData_df, function(x) is.numeric(x) || is.integer(x))
countData_num <- countData_df[, numeric_cols_idx, drop = FALSE]

genes_of_interest <- c(
  "HOXA1","HOXA10","HOXA11","HOXA11OS","HOXA2","HOXA3","HOXA4","HOXA5","HOXA6","HOXA7","HOXA9",
  "HOXB1","HOXB2","HOXB3","HOXB3OS","HOXB4","HOXB5","HOXB5OS","HOXB6","HOXB7","HOXB8","HOXB9",
  "HOXC10","HOXC11","HOXC13","HOXC4","HOXC5","HOXC6","HOXC8","HOXC9",
  "HOXD10","HOXD11","HOXD3","HOXD3OS1","HOXD8","HOXD9",
  "PAX3","PAX6",
  "POU2F1","POU2F2","POU2F3",
  "POU3F1","POU3F2","POU3F3","POU3F4",
  "POU4F1","POU4F3",
  "POU5F2",
  "POU6F1"
)

row_idx <- match(genes_of_interest, rownames(countData_num))
row_idx <- row_idx[!is.na(row_idx)]
expr_sub <- countData_num[row_idx, , drop = FALSE]

expr_sub_num <- as.data.frame(
  lapply(expr_sub, function(col) as.numeric(as.character(col))),
  check.names = FALSE
)
expr_sub_num <- as.matrix(expr_sub_num)
rownames(expr_sub_num) <- rownames(expr_sub)
colnames(expr_sub_num) <- colnames(expr_sub)

## --- updated grouping logic (no genotype) ---

sample_names <- colnames(expr_sub_num)

# only label condition from colnames (Control vs LPS)
cond_vec <- ifelse(grepl("LPS", sample_names, ignore.case = TRUE), "LPS",
            ifelse(grepl("CTRL|CONTROL", sample_names, ignore.case = TRUE), "Control", "Unknown"))

# group_key is just condition now
group_key <- cond_vec

cat("Detected conditions:\n")
print(unique(group_key))

collapse_by_key <- function(mat, keys) {
  split_cols <- split(seq_along(keys), keys)
  collapsed <- sapply(split_cols, function(idx) {
    rowMeans(mat[, idx, drop = FALSE], na.rm = TRUE)
  })
  as.matrix(collapsed)
}

expr_collapsed <- collapse_by_key(expr_sub_num, group_key)

# Ensure column names are just "Control", "LPS"
colnames(expr_collapsed) <- unique(group_key)[match(colnames(expr_collapsed), unique(group_key))]

cat("Collapsed matrix dim:", paste(dim(expr_collapsed), collapse=" x "), "\n")
print(colnames(expr_collapsed))

###########################
## Build annotation (Condition only)
###########################
annotation_col <- data.frame(
  Condition = colnames(expr_collapsed),
  stringsAsFactors = FALSE
)
rownames(annotation_col) <- colnames(expr_collapsed)

ann_colors <- list(
  Condition = c(
    Control = "#8da0cb",
    LPS     = "#e78ac3",
    Unknown = "grey90"
  )
)

###########################
## Color palettes
###########################
hm_diverging <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

hm_sequential <- colorRampPalette(
  c("#08306B", "#2879B9", "#72B2D7", "#FDD67B", "#F03B20")
)(200)

###########################
## Heatmap A: absolute expression
###########################
heat_abs <- pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "none",
  color = hm_sequential,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 6.5,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 28,
  border_color = NA,
  legend = TRUE,
  main = "Absolute expression (Control vs LPS)",
  treeheight_row = 10,
  treeheight_col = 12
)

###########################
## Heatmap B: row-scaled (per-gene z-score)
###########################
heat_rel <- pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "row",
  color = hm_diverging,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 6.5,
  fontsize_col = 9,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 28,
  border_color = NA,
  legend = TRUE,
  main = "Row-scaled z-score (Control vs LPS)",
  treeheight_row = 10,
  treeheight_col = 12
)

###########################
## Combine into one figure and save as PNG
###########################
g_abs <- heat_abs[[4]]
g_rel <- heat_rel[[4]]

grid.newpage()
grid.arrange(g_abs, g_rel, ncol = 2)

png("HOX_PAX_POU_Control_vs_LPS_dual_heatmap.png", width = 2400, height = 1200, res = 300)
grid.arrange(g_abs, g_rel, ncol = 2)

cat("Saved combined figure 'HOX_PAX_POU_Control_vs_LPS_dual_heatmap.png'\n")

# Automatically added: save last plot as SVG
ggsave('figure_121.svg', width = 8, height = 6)


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
})

###########################
## 1. Derive group information from column names
###########################
sample_names <- colnames(expr_collapsed)

annotation_col <- data.frame(
  Genotype = ifelse(
    grepl("WT",  sample_names, ignore.case = TRUE), "wt",
    ifelse(grepl("MUT", sample_names, ignore.case = TRUE), "mut", "Unknown")
  ),
  Condition = ifelse(
    grepl("LPS", sample_names, ignore.case = TRUE), "LPS",
    ifelse(grepl("CTRL|CONTROL", sample_names, ignore.case = TRUE), "Control", "Unknown")
  ),
  stringsAsFactors = FALSE
)

rownames(annotation_col) <- sample_names

###########################
## 2. Drop Genotype annotation if it's all 'Unknown'
###########################
if (length(unique(annotation_col$Genotype)) == 1 && unique(annotation_col$Genotype) == "Unknown") {
  annotation_col$Genotype <- NULL
}

###########################
## 3. Define annotation colors (only for the columns that exist)
###########################
ann_colors <- list()

if ("Genotype" %in% colnames(annotation_col)) {
  ann_colors$Genotype <- c(
    wt      = "#b3de69",  # light green
    mut     = "#bc80bd",  # mauve / purple
    Unknown = "grey80"
  )
}

if ("Condition" %in% colnames(annotation_col)) {
  ann_colors$Condition <- c(
    Control = "#8da0cb",  # periwinkle
    LPS     = "#e78ac3",  # pink
    Unknown = "grey90"
  )
}

###########################
## 4. Heatmap color palette (diverging for z-score)
###########################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

###########################
## 5. Plot (publication-style)
###########################
pheatmap(
  expr_collapsed,
  cluster_rows = FALSE,
  cluster_cols = TRUE,
  scale = "row",                 # z-score per gene (relative up/down)
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,

  # visual polish
  fontsize_row = 7,              # gene name size
  fontsize_col = 10,             # condition/genotype label size
  angle_col = 45,                # tilt column labels
  cellheight = 10,               # row spacing
  cellwidth  = 28,               # wider columns for readability
  border_color = NA,             # remove grid lines
  legend = TRUE,
  main = "HOX / PAX / POU Gene Expression (row-scaled)",
  treeheight_row = 10,
  treeheight_col = 12
)

# Automatically added: save last plot as SVG
ggsave('figure_122.svg', width = 8, height = 6)


In [ ]:
countdata45 <- normalized_counts_1

In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
})

###########################
## 1. Derive group information from column names
###########################
sample_names <- colnames(expr_collapsed)

annotation_col <- data.frame(
  Genotype = ifelse(grepl("WT",  sample_names, ignore.case = TRUE), "wt",
             ifelse(grepl("MUT", sample_names, ignore.case = TRUE), "mut", "Unknown")),
  Condition = ifelse(grepl("LPS", sample_names, ignore.case = TRUE), "LPS",
               ifelse(grepl("CTRL|CONTROL", sample_names, ignore.case = TRUE), "Control", "Unknown")),
  stringsAsFactors = FALSE
)
rownames(annotation_col) <- sample_names

###########################
## 2. Decide whether Genotype is meaningful
##    (If everything is "Unknown", drop that column from annotation)
###########################
if (length(unique(annotation_col$Genotype)) == 1 && unique(annotation_col$Genotype) == "Unknown") {
  annotation_col$Genotype <- NULL
}

###########################
## 3. Define annotation colors
###########################
ann_colors <- list()

if ("Genotype" %in% colnames(annotation_col)) {
  ann_colors$Genotype <- c(
    wt       = "#b3de69",  # light green
    mut      = "#bc80bd",  # mauve/purple
    Unknown  = "grey80"
  )
}

if ("Condition" %in% colnames(annotation_col)) {
  ann_colors$Condition <- c(
    Control = "#8da0cb",   # periwinkle
    LPS     = "#e78ac3",   # pink
    Unknown = "grey90"
  )
}

###########################
## 4. Heatmap color palette
##    We'll keep the same diverging palette you used
##    for row-scaled visualization.
###########################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

###########################
## 5. Plot heatmap with improved aesthetics
###########################
pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "row",                 # per-gene z-score to show up/down pattern
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,

  # visual polish:
  fontsize_row = 7,              # gene label size
  fontsize_col = 10,             # column label size
  angle_col = 45,                # tilt x labels
  cellheight = 10,               # vertical spacing per gene
  cellwidth  = 28,               # wider columns for readability
  border_color = NA,             # remove box lines/grid
  legend = TRUE,
  main = "HOX / PAX / POU Gene Expression (row-scaled)",
  treeheight_row = 10,
  treeheight_col = 12
)

# Automatically added: save last plot as SVG
ggsave('figure_124.svg', width = 8, height = 6)


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
  library(stringr)
})

###########################
## 1. Derive group information from column names
###########################
sample_names <- colnames(expr_collapsed)

annotation_col <- data.frame(
  Genotype = ifelse(grepl("WT",  sample_names, ignore.case = TRUE), "wt",
             ifelse(grepl("MUT", sample_names, ignore.case = TRUE), "mut", "Unknown")),
  Condition = ifelse(grepl("LPS", sample_names, ignore.case = TRUE), "LPS",
               ifelse(grepl("CTRL|CONTROL", sample_names, ignore.case = TRUE), "Control", "Unknown")),
  stringsAsFactors = FALSE
)
rownames(annotation_col) <- sample_names

###########################
## 2. Decide whether Genotype is meaningful
##    (If everything is "Unknown", drop that column from annotation)
###########################
if (length(unique(annotation_col$Genotype)) == 1 && unique(annotation_col$Genotype) == "Unknown") {
  annotation_col$Genotype <- NULL
}

###########################
## 3. Define annotation colors
###########################
ann_colors <- list()

if ("Genotype" %in% colnames(annotation_col)) {
  ann_colors$Genotype <- c(
    wt       = "#b3de69",  # light green
    mut      = "#bc80bd",  # mauve/purple
    Unknown  = "grey80"
  )
}

if ("Condition" %in% colnames(annotation_col)) {
  ann_colors$Condition <- c(
    Control = "#8da0cb",   # periwinkle
    LPS     = "#e78ac3",   # pink
    Unknown = "grey90"
  )
}

###########################
## 4. Heatmap color palette
##    We'll keep the same diverging palette you used
##    for row-scaled visualization.
###########################
hm_colors <- colorRampPalette(
  c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026")
)(200)

###########################
## 5. Plot heatmap with improved aesthetics
###########################
pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "row",                 # per-gene z-score to show up/down pattern
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,

  # visual polish:
  fontsize_row = 7,              # gene label size
  fontsize_col = 10,             # column label size
  angle_col = 45,                # tilt x labels
  cellheight = 10,               # vertical spacing per gene
  cellwidth  = 28,               # wider columns for readability
  border_color = NA,             # remove box lines/grid
  legend = TRUE,
  main = "HOX / PAX / POU Gene Expression (row-scaled)",
  treeheight_row = 10,
  treeheight_col = 12
)

# Automatically added: save last plot as SVG
ggsave('figure_125.svg', width = 8, height = 6)


In [ ]:
###########################
## Load packages
###########################
suppressPackageStartupMessages({
  library(pheatmap)
})

###########################
## 1️⃣ Annotation for groups
###########################
annotation_col <- data.frame(
  Genotype = c("WT", "WT", "MUT", "MUT"),
  Condition = c("Control", "LPS", "Control", "LPS")
)
rownames(annotation_col) <- colnames(expr_collapsed)

###########################
## 2️⃣ Define annotation colors
###########################
ann_colors <- list(
  Genotype = c(WT = "#66c2a5", MUT = "#fc8d62"),
  Condition = c(Control = "#8da0cb", LPS = "#e78ac3")
)

###########################
## 3️⃣ Nice color palette
###########################
hm_colors <- colorRampPalette(c("#313695", "#74add1", "#ffffbf", "#f46d43", "#a50026"))(200)

###########################
## 4️⃣ Save as JPEG
###########################
jpeg("HOX_PAX_POU_LMC_heatmap.jpg", width = 2000, height = 2000, res = 300)

pheatmap(
  expr_collapsed,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  scale = "row",
  color = hm_colors,
  annotation_col = annotation_col,
  annotation_colors = ann_colors,
  show_rownames = TRUE,
  show_colnames = TRUE,
  fontsize_row = 7,
  fontsize_col = 10,
  angle_col = 45,
  cellheight = 10,
  cellwidth  = 25,
  border_color = NA,
  legend = TRUE,
  main = "HOX / PAX / POU Gene Expression (LMC)",
  treeheight_row = 10
)

cat("✅ Heatmap saved as 'HOX_PAX_POU_LMC_heatmap.jpg' in your working directory.\n")

# Automatically added: save last plot as SVG
ggsave('figure_126.svg', width = 8, height = 6)


In [ ]:

# Convert row names to uppercase to match case insensitivity
rownames(countData) <- toupper(rownames(countData))
rownames(res2_I_sig) <- toupper(rownames(res2_I_sig))
rownames(res2_II_sig) <- toupper(rownames(res2_II_sig))
nf_kb$Gene_name <- toupper(nf_kb$Gene_name)  # Ensuring case consistency

# Intersect gene names across different datasets
common_genes_I_sig <- intersect(rownames(countData), rownames(res2_I_sig))
common_genes_II_sig <- intersect(rownames(countData), rownames(res2_II_sig))
common_genes_nf_kb <- intersect(rownames(countData), nf_kb$Gene_name)

# Combine intersections to find genes common across all conditions
common_genes_final <- Reduce(intersect, list(common_genes_I_sig, common_genes_II_sig, common_genes_nf_kb))

# Filter countData to include only rows that have genes present in all intersections
filteredCountData <- countData[common_genes_final, ]

# Further filter rows to exclude certain gene name prefixes
filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredCountData), perl = TRUE)
filteredCountDataFiltered <- filteredCountData[filtered_rows, ]

# Calculate mean expression levels
meanExpressionLevels <- rowMeans(filteredCountDataFiltered)

# Sort genes by mean expression levels in decreasing order and select the top 50
top50Genes <- names(sort(meanExpressionLevels, decreasing = TRUE))#[1:50]

# Filter the data to include only these top 50 genes
top50FilteredData <- filteredCountDataFiltered[top50Genes, ]

# Optional: Define or calculate any specific annotation for columns if needed
# manual_annotation <- data.frame(Condition = c("Control", "Treatment"), row.names = colnames(top50FilteredData))



# Plot the heatmap of the top 50 filtered countData
pheatmap(top50FilteredData,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
         fontsize_row = 6,
         main = "")
# Automatically added: save last plot as SVG
ggsave('figure_127.svg', width = 8, height = 6)


In [ ]:
# Number of genes to label
N <- 10  

# Sorting the data frame based on the criteria (e.g., padj)
sorted_indices <- order(res2_III$padj)

# Initializing labels to NA
res2_III$labels <- NA

# Selecting top N genes based on sorted indices
top_N_indices <- sorted_indices[1:N]

# Identifying those among the top N that need to be filtered out
filtered_indices <- top_N_indices[grepl("^Gm|-ik$|Mir|^Rpl", res2_III$SYMBOL[top_N_indices])]

# Identifying how many more genes we need to pick
num_more <- length(filtered_indices)

# Picking the next top-most genes that are valid
next_valid_indices <- sorted_indices[(N + 1):(N + num_more)]
next_valid_indices <- next_valid_indices[!grepl("^Gm|-ik$|Mir|^Rpl", res2_III$SYMBOL[next_valid_indices])]

# Combining the valid top indices and the next valid indices to get top N labels
final_top_N_indices <- c(setdiff(top_N_indices, filtered_indices), next_valid_indices[1:num_more])

# Filtering out NA indices
final_top_N_indices <- final_top_N_indices[!is.na(final_top_N_indices)]
final_top_N_indices <- final_top_N_indices[!is.na(res2_III$SYMBOL[final_top_N_indices])]

# Checking if any of the final_top_N_indices are NA or if corresponding SYMBOLs are NA
if (any(is.na(final_top_N_indices)) || any(is.na(res2_III$SYMBOL[final_top_N_indices]))) {
    stop("NA indices detected. Please resolve before proceeding.")
}

# Setting the labels for these final top N indices
res2_III$labels[final_top_N_indices] <- res2_III$SYMBOL[final_top_N_indices]

# Creating the EnhancedVolcano plot
EnhancedVolcano(res2_III,
                lab = rownames(res2_III),
                x = 'log2FoldChange',
                y = 'padj')

# Automatically added: save last plot as SVG
ggsave('figure_128.svg', width = 8, height = 6)


In [ ]:
res2_I_df <- data.frame(res2_I)
res2_II_df <- data.frame(res2_II)
res2_III_df <- data.frame(res2_III)

In [ ]:
res2_I_full <- data.frame(res2_I_full)
res2_II_full <- data.frame(res2_II_full)

In [ ]:
colnames(res2_II_full)

In [ ]:
lmc_data <- read.csv('merged_LMC_I_II_III_deseq2results.csv')

In [ ]:
res2_I_df$gene <- rownames(res2_I_df)
res2_II_df$gene <- rownames(res2_II_df)


merged_data_mef <- merge(res2_I_df, res2_II_df, by="gene", suffixes = c("_I", "_II"))

res2_III_df <- data.frame(res2_III)
res2_III_df$gene <- rownames(res2_III_df)
mef_data <- merge(merged_data_mef, res2_III_df, by="gene", suffixes = c("", "_III"))


mef_data <- mef_data %>%
  rename_with(.fn = ~paste0(., "_III"), .cols = tail(names(mef_data), 6))



In [ ]:
library(ggplot2)

In [ ]:
ggplot(res2_II_df, aes(x = baseMean, y = log2FoldChange)) +
  geom_point(alpha = 0.6) + # Set alpha for better visualization if points overlap
  scale_x_log10() + # Log-transform the baseMean axis for better visualization
  theme_minimal() +
  labs(x = "Base Mean (log scale)", 
       y = "Log2 Fold Change", 
       title = "Scatter Plot of baseMean vs. log2FoldChange") +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") # Add a horizontal line at y=0
# Automatically added: save last plot as SVG
ggsave('figure_135.svg', width = 8, height = 6)


In [ ]:
ggplot(res2_I_df, aes(x = baseMean, y = log2FoldChange)) +
  geom_point(alpha = 0.6) + # Set alpha for better visualization if points overlap
  scale_x_log10() + # Log-transform the baseMean axis for better visualization
  theme_minimal() +
  labs(x = "Base Mean (log scale)", 
       y = "Log2 Fold Change", 
       title = "Scatter Plot of baseMean vs. log2FoldChange") +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red")
# Automatically added: save last plot as SVG
ggsave('figure_136.svg', width = 8, height = 6)


In [ ]:
library(ggplot2)

# Assuming your datasets are read into data frames named `lmc_data` and `mef_data`
# You can read your data with something like:
# lmc_data <- read.csv("path_to_your_lmc_data_file.csv")
# mef_data <- read.csv("path_to_your_mef_data_file.csv")

# 1. Comparison of I conditions between lmc_data and mef_data
ggplot() +
  geom_point(data = mef_data, aes(x = baseMean_I, y = log2FoldChange_I), color = "blue", alpha = 0.5) +
  geom_point(data = lmc_data, aes(x = baseMean_I, y = log2FoldChange_I), color = "red", alpha = 0.5) +
  labs(title = "Comparison of Condition I: MEF vs LMC", x = "Base Mean", y = "Log2 Fold Change") +
  scale_color_manual(values = c("MEF" = "blue", "LMC" = "red")) +
  theme_minimal()


# Automatically added: save last plot as SVG
ggsave('figure_137.svg', width = 8, height = 6)


In [ ]:
# Load necessary library for plotting
library(ggplot2)

# Plotting
ggplot(mef_data, aes(x = log2FoldChange_I, y = log2FoldChange_II)) +
  geom_point() +  # Add points
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") + # Add y=x line for reference
  theme_minimal() +  # Use a minimal theme
  labs(x = "Log2 Fold Change (res2_I)", y = "Log2 Fold Change (res2_II)", 
       title = "Comparison of Log2 Fold Changes between res2_I and res2_II") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))  # Improve label readability
# Automatically added: save last plot as SVG
ggsave('figure_138.svg', width = 8, height = 6)


In [ ]:
# Load necessary library for plotting
library(ggplot2)

# Plotting
ggplot(lmc_data, aes(x = log2FoldChange_I, y = log2FoldChange_II)) +
  geom_point() +  # Add points
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") + # Add y=x line for reference
  theme_minimal() +  # Use a minimal theme
  labs(x = "Log2 Fold Change (res2_I)", y = "Log2 Fold Change (res2_II)", 
       title = "Comparison of Log2 Fold Changes between res2_I and res2_II") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))  # Improve label readability
# Automatically added: save last plot as SVG
ggsave('figure_139.svg', width = 8, height = 6)


In [ ]:
# 2. Comparison of II conditions between lmc_data and mef_data
ggplot() +
  geom_point(data = mef_data, aes(x = baseMean_II, y = log2FoldChange_II), color = "blue", alpha = 0.5) +
  geom_point(data = lmc_data, aes(x = baseMean_II, y = log2FoldChange_II), color = "red", alpha = 0.5) +
  labs(title = "Comparison of Condition II: MEF vs LMC", x = "Base Mean", y = "Log2 Fold Change") +
  scale_color_manual(values = c("MEF" = "blue", "LMC" = "red")) +
  theme_minimal()
# Automatically added: save last plot as SVG
ggsave('figure_140.svg', width = 8, height = 6)


In [ ]:
ggplot(mef_data, aes(x = log2FoldChange_I, y = log2FoldChange_II)) +
  geom_point(alpha = 0.5) + # Plot points with semi-transparency for better visualization
  geom_abline(slope = 1, intercept = 0, color = "red", linetype = "dashed") + # Add diagonal line
  labs(title = "MEF Data: Comparison of Log2 Fold Change between Conditions I and II", 
       x = "Log2 Fold Change I", 
       y = "Log2 Fold Change II") +
  theme_minimal() # Use minimal theme for a clean look
# Automatically added: save last plot as SVG
ggsave('figure_141.svg', width = 8, height = 6)


In [ ]:
ggplot(lmc_data, aes(x = log2FoldChange_I, y = log2FoldChange_II)) +
  geom_point(alpha = 0.5) + # Plot points with semi-transparency for better visualization
  geom_abline(slope = 1, intercept = 0, color = "red", linetype = "dashed") + # Add diagonal line
  labs(title = "LMC Data: Comparison of Log2 Fold Change between Conditions I and II", 
       x = "Log2 Fold Change I", 
       y = "Log2 Fold Change II") +
  theme_minimal() # Use minimal theme for a clean look
# Automatically added: save last plot as SVG
ggsave('figure_142.svg', width = 8, height = 6)


In [ ]:
comparison_data <- merge(mef_data, lmc_data, by = "gene", suffixes = c("_MEF", "_LMC"))

# Now plot baseMean_I from MEF vs baseMean_I from LMC
ggplot(comparison_data, aes(x = log2FoldChange_III_MEF, y = log2FoldChange_III_LMC)) +
  geom_point(alpha = 0.5) +  # Adjust opacity if points are too dense
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +  # Add a y=x reference line
  labs(title = "Comparison of log2 Fold Change III between MEF and LMC", x = "Log2 FoldChange III MEF", y = "Log2 FoldChange III LMC") +
  theme_minimal()
# Automatically added: save last plot as SVG
ggsave('figure_143.svg', width = 8, height = 6)


In [ ]:
# Loading the DE genes from analysis I and II
gene_list_I <- na.omit(rownames(res2_I))  # Remove NAs
gene_list_II <- na.omit(rownames(res2_II))  # Remove NAs

# Making a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
#Reading the list of DE genes from analysis I,II,III,IV,V respectively and merge them together for next analyses.

DE_I <- data.frame(rownames(res2_I))
DE_II <- data.frame(rownames(res2_II))
DE_III <- data.frame(rownames(res2_III))


colnames(DE_I)[1] <- "Gene_name"
colnames(DE_II)[1] <- "Gene_name"
colnames(DE_III)[1] <- "Gene_name"




In [ ]:
combined_column <- unique(data.frame(c(DE_I$Gene_name,DE_II$Gene_name,DE_III$Gene_name)))
colnames(combined_column)[1] <- "Gene_name"
length(combined_column$Gene_name)

In [ ]:
write.csv(combined_column,"MEF_DE_genes_total_Jan15_2024.csv")

Carrying out gene-set enrichment analysis as highlighted in the following link : https://learn.gencore.bio.nyu.edu/rna-seq-analysis/gene-set-enrichment-analysis/

In [ ]:
# Setting the organism mouse
organism = "org.Mm.eg.db"
library(organism, character.only = TRUE)

In [ ]:
#Reformatting the list of genes for the plot
original_gene_list1 <- res2_I$log2FoldChange
names(original_gene_list1) <- rownames(res2_I)
gene_list1<-na.omit(original_gene_list1)
# sorting the list in decreasing order (required for clusterProfiler)
gene_list1 = sort(gene_list1, decreasing = TRUE)

#Reformatting the list of genes for the plot
original_gene_list2 <- res2_II$log2FoldChange
names(original_gene_list2) <- rownames(res2_II)
gene_list2<-na.omit(original_gene_list2)
# sorting the list in decreasing order (required for clusterProfiler)
gene_list2 = sort(gene_list2, decreasing = TRUE)

#Reformatting the list of genes for the plot
original_gene_list3 <- res2_III$log2FoldChange
names(original_gene_list3) <- rownames(res2_III)
gene_list3<-na.omit(original_gene_list3)
# sorting the list in decreasing order (required for clusterProfiler)
gene_list3 = sort(gene_list3, decreasing = TRUE)

In [ ]:
gse1 <- gseGO(geneList=gene_list1, 
             ont ="BP", 
             keyType = "SYMBOL", 
             nPerm = 10000, 
             minGSSize = 3, 
             maxGSSize = 800, 
             pvalueCutoff = 0.2, 
             verbose = TRUE, 
             OrgDb = organism, 
             pAdjustMethod = "BH")

gse2 <- gseGO(geneList=gene_list2, 
             ont ="BP", 
             keyType = "SYMBOL", 
             nPerm = 10000, 
             minGSSize = 3, 
             maxGSSize = 800, 
             pvalueCutoff = 0.2, 
             verbose = TRUE, 
             OrgDb = organism, 
             pAdjustMethod = "BH")


gse3 <- gseGO(geneList=gene_list3, 
             ont ="BP", 
             keyType = "SYMBOL", 
             nPerm = 10000, 
             minGSSize = 3, 
             maxGSSize = 800, 
             pvalueCutoff = 0.2, 
             verbose = TRUE, 
             OrgDb = organism, 
             pAdjustMethod = "BH")



In [ ]:
gse_termsim1 <- pairwise_termsim(gse1)
gse_termsim2 <- pairwise_termsim(gse2)
gse_termsim3 <- pairwise_termsim(gse3)

In [ ]:
genelist_gse3 <- data.frame(gse3@geneList)
colnames(genelist_gse3)[1] <- "FoldChange"

write.csv(genelist_gse3,'genelist_III_MEF_gsea_results.csv')

In [ ]:
require(DOSE)
dotplot(gse1, showCategory=20, split=".sign") + facet_grid(.~.sign)
dotplot(gse2, showCategory=20, split=".sign") + facet_grid(.~.sign)
dotplot(gse3, showCategory=20, split=".sign") + facet_grid(.~.sign)
# Automatically added: save last plot as SVG
ggsave('figure_155.svg', width = 8, height = 6)


In [ ]:
GO_I_mef <- data.frame(gse1@result)
GO_II_mef <- data.frame(gse2@result)
GO_III_mef <- data.frame(gse3@result)

In [ ]:
write.csv(countData2,'MEF_May13_2025_analysis_counts.csv')

In [ ]:
View(as.data.frame(gse1))

In [ ]:
gse1@result$Description <- sub("^(\\w)", "\\U\\1", gse1@result$Description, perl = TRUE)

In [ ]:
dotplot(gse1,showCategory=c("Cellular response to interferon-beta","Antimicrobial humoral response","Cellular response to lipopolysaccharide","Neutrophil migration","Embryonic skeletal system morphogenesis", "B cell receptor signaling pathway","Response to lipopolysaccharide"),split=".sign")+ facet_grid(.~.sign)
# Automatically added: save last plot as SVG
ggsave('figure_160.svg', width = 8, height = 6)


In [ ]:
write.csv(GO_I_mef,'mef_GO_I_GSEA_clusterprofiler.csv')
write.csv(GO_II_mef,'mef_GO_II_GSEA_clusterprofiler.csv')
write.csv(GO_III_mef,'mef_GO_III_GSEA_clusterprofiler.csv')


In [ ]:
gse2@result$Description <- sub("^(\\w)", "\\U\\1", gse2@result$Description, perl = TRUE)

In [ ]:
View(as.data.frame(gse2))

In [ ]:
gse3@result$Description <- sub("^(\\w)", "\\U\\1", gse3@result$Description, perl = TRUE)

In [ ]:
dotplot(gse3,showCategory=c("Spindle checkpoint signaling","Peripheral nervous system neuron differentiation","Negative regulation of chromosome segregation","Innate immune response","Response to external biotic stimulus","biological process involved in interspecies interaction between organisms","Plasma membrane invagination","Hemopoiesis","Embryonic morphogenesis","Medium-chain fatty acid metabolic process"),split=".sign")+ facet_grid(.~.sign)
# Automatically added: save last plot as SVG
ggsave('figure_165.svg', width = 8, height = 6)


In [ ]:
p <- dotplot(
  gse3,
  showCategory = c(
    "Spindle checkpoint signaling",
    "Peripheral nervous system neuron differentiation",
    "Negative regulation of chromosome segregation",
    "Innate immune response",
    "Response to external biotic stimulus",
    "biological process involved in interspecies interaction between organisms",
    "Plasma membrane invagination",
    "Hemopoiesis",
    "Embryonic morphogenesis",
    "Medium-chain fatty acid metabolic process"
  ),
  split = ".sign"
) + 
  facet_grid(. ~ .sign) +
  theme(
    aspect.ratio = 2,     # > 1 makes it taller; try 2 if you want very tall
    plot.margin = margin(20, 20, 20, 20)  # extra padding
  )
# Automatically added: save last plot as SVG
ggsave('figure_166.svg', width = 8, height = 6)


In [ ]:
p

In [ ]:
svg("dotplot_mef_gse3_sep30.svg", width = 10, height = 12)
print(p)
dev.off()


In [ ]:
dotplot(gse3,showCategory=c("Spindle checkpoint signaling","Peripheral nervous system neuron differentiation","Negative regulation of chromosome segregation","Innate immune response","Response to external biotic stimulus","biological process involved in interspecies interaction between organisms","Plasma membrane invagination","Hemopoiesis","Embryonic morphogenesis","Medium-chain fatty acid metabolic process"),split=".sign")+ facet_grid(.~.sign)
# Automatically added: save last plot as SVG
ggsave('figure_169.svg', width = 8, height = 6)


In [ ]:
# Assuming `p` is your ggplot object
p <- ridgeplot(gse1) +
  labs(x = "enrichment distribution") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),  # Rotate and reduce text size
    axis.text.y = element_text(size = 6),  # Further reduce text size for y-axis
    strip.text.y = element_text(size = 6, angle = 0)  # Reduce facet label size and optionally rotate
  ) +
  coord_fixed(ratio = 0.7)  # Adjust aspect ratio

p <- ridgeplot(gse2) +
  labs(x = "enrichment distribution") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),  # Rotate and reduce text size
    axis.text.y = element_text(size = 6),  # Further reduce text size for y-axis
    strip.text.y = element_text(size = 6, angle = 0)  # Reduce facet label size and optionally rotate
  ) +
  coord_fixed(ratio = 0.7)  # Adjust aspect ratio


p <- ridgeplot(gse3) +
  labs(x = "enrichment distribution") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),  # Rotate and reduce text size
    axis.text.y = element_text(size = 6),  # Further reduce text size for y-axis
    strip.text.y = element_text(size = 6, angle = 0)  # Reduce facet label size and optionally rotate
  ) +
  coord_fixed(ratio = 0.7)  # Adjust aspect ratio



# Automatically added: save last plot as SVG
ggsave('figure_171.svg', width = 8, height = 6)


In [ ]:
p1 <- gseaplot(gse1, geneSetID = 1, by = "runningScore", title = gse1$Description[1])
p2 <- gseaplot(gse1, geneSetID = 1, by = "preranked", title = gse1$Description[1])
#p3 <- gseaplot(gse1, geneSetID = 1, title = gse1$Description[1])
cowplot::plot_grid(p1, p2, ncol=1, labels=LETTERS[1:2])
# Automatically added: save last plot as SVG
ggsave('figure_172.svg', width = 8, height = 6)


In [ ]:
gseaplot2(gse1, geneSetID = 1, title = gse1$Description[1])

In [ ]:
p1 <- gseaplot(gse2, geneSetID = 1, by = "runningScore", title = gse2$Description[1])
p2 <- gseaplot(gse2, geneSetID = 1, by = "preranked", title = gse2$Description[1])
cowplot::plot_grid(p1, p2, ncol=1, labels=LETTERS[1:2])
# Automatically added: save last plot as SVG
ggsave('figure_174.svg', width = 8, height = 6)


In [ ]:
gseaplot2(gse2, geneSetID = 1, title = gse2$Description[1])

In [ ]:
p1 <- gseaplot(gse3, geneSetID = 1, by = "runningScore", title = gse3$Description[1])
p2 <- gseaplot(gse3, geneSetID = 1, by = "preranked", title = gse3$Description[1])
cowplot::plot_grid(p1, p2, ncol=1, labels=LETTERS[1:2])
# Automatically added: save last plot as SVG
ggsave('figure_176.svg', width = 8, height = 6)


In [ ]:
gseaplot2(gse3, geneSetID = 1, title = gse3$Description[1])

In [ ]:
enriched_terms_df_gse1 <- gse1@result

# View the first few rows of the data frame
head(enriched_terms_df_gse1)

In [ ]:
enriched_terms_df_gse2 <- gse2@result

# View the first few rows of the data frame
head(enriched_terms_df_gse2)

In [ ]:
enriched_terms_df_gse3 <- gse3@result

# View the first few rows of the data frame
head(enriched_terms_df_gse3)

In [ ]:
homeobox <- read.csv('homebox_gene_list.csv',header = FALSE)
colnames(homeobox) <- "Gene_name"

In [ ]:
homeobox$Gene_name <- toupper(homeobox$Gene_name)

In [ ]:
# Load necessary libraries
library(pheatmap)
library(dplyr)

# Ensure the gene names in homeobox are present in the row names of countData
common_genes <- intersect(homeobox$Gene_name, rownames(countData))

# Filter countData to include only rows that have genes present in common_genes
filteredCountData_homeo <- countData[common_genes, ]

# Define the width and height of the plot
# Increase the height to ensure all rows are visible
plot_width <- 20
plot_height <- 100

# Save the heatmap as a PDF
#pdf("Heatmap_of_Homeobox_Gene_Expression.pdf", width = plot_width, height = plot_height)

# Plot the heatmap of the filtered countData with clustering and improved row names visibility
pheatmap(filteredCountData_homeo,
         cluster_rows = TRUE,  # Cluster by rows
         cluster_cols = FALSE,
         annotation_col = as.data.frame(manual_annotation),  # Ensure this is defined or modify as needed
         show_rownames = TRUE,
         show_colnames = FALSE,  # Hiding column names
         scale = "row",
       #  fontsize_row = 4,  # Adjust the font size for row names
      #   fontsize_col = 10,  # Adjust the font size for column names if shown
         main = "Heatmap of Homeobox Gene Expression",
         fontsize = 2,  # Increase overall font size
         cellheight = 2)  # Set the height of each cell (row)


# Automatically added: save last plot as SVG
ggsave('figure_183.svg', width = 8, height = 6)


In [ ]:
nf_kb <- read.table('nf_kb_sites.txt')

In [ ]:
colnames(nf_kb)[1] <- "Gene_name"

In [ ]:
library(clusterProfiler)
library(org.Mm.eg.db)

In [ ]:
gene_listI <- rownames(res2_I)
entrez_ids <- mapIds(org.Mm.eg.db, keys = gene_listI, column = "ENTREZID", keytype = "SYMBOL")

# Running KEGG enrichment analysis using the Entrez IDs
kegg_results <- enrichKEGG(gene = entrez_ids, organism = 'mmu', keyType = 'kegg')
# Filter to get genes related to NF-κB signaling pathway (mmu04064)
nfkb_genes <- kegg_results$geneID[kegg_results$ID == 'mmu04064']
# Convert the gene list from a comma-separated string to a vector
nfkb_genes_list <- unlist(strsplit(nfkb_genes, split = "/"))
# Convert Entrez IDs back to Gene Names
nfkb_gene_names <- mapIds(org.Mm.eg.db, keys = nfkb_genes_list, column = "SYMBOL", keytype = "ENTREZID")
nf_I <- data.frame(nfkb_gene_names)

In [ ]:
#This list of genes with Tata,GC boxes have been obtained from Homer analysis

In [ ]:
library(VennDiagram)
library(grid)

# Loading the DE genes from analysis I and II
gene_list_I <- na.omit(rownames(res2_I_up))  # Remove NAs
gene_list_II <- na.omit(rownames(res2_II_up))  # Remove NAs

# Making a list of the gene lists in flipped order
gene_lists <- list(gene_list_II, gene_list_I)

# Run the Venn diagram function with flipped positions, no labels, and no numbers
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set category names to empty strings to hide labels
    output = TRUE,
    fill = c("blue", "red"),  # Flipped the colors to match the flipped order
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    rotation.degree = 180,  # Rotate the diagram by 180 degrees
    reverse = TRUE,  # Reverse the order of categories
    cex = 0  # Remove the numbers in circles
)

# Plot the diagram
grid.draw(venn.plot)





In [ ]:
library(VennDiagram)
library(grid)

# Loading the DE genes from analysis I and II
gene_list_I <- na.omit(rownames(res2_I_down))  # Remove NAs
gene_list_II <- na.omit(rownames(res2_II_down))  # Remove NAs

# Making a list of the gene lists in flipped order
gene_lists <- list(gene_list_II, gene_list_I)

# Run the Venn diagram function with flipped positions, no labels, and no numbers
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set category names to empty strings to hide labels
    output = TRUE,
    fill = c("blue", "red"),  # Flipped the colors to match the flipped order
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    rotation.degree = 180,  # Rotate the diagram by 180 degrees
    reverse = TRUE,  # Reverse the order of categories
    cex = 0,  # Remove the numbers in circles
    cat.cex = 0,  # Remove category labels
    label.col = c("transparent", "transparent", "transparent")  # Make labels transparent
)

# Plot the diagram
grid.draw(venn.plot)


In [ ]:
# Loading the DE genes from analysis I and II
gene_list_I <- na.omit(rownames(res2_I_up))  # Remove NAs
gene_list_II <- na.omit(rownames(res2_II_up))  # Remove NAs

# Making a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = NA,  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
library(VennDiagram)
library(grid)

# Loading the DE genes from analysis I and II
gene_list_I <- na.omit(rownames(res2_I_up))  # Remove NAs
gene_list_II <- na.omit(rownames(res2_II_up))  # Remove NAs

# Making a list of the gene lists in flipped order
gene_lists <- list(gene_list_II, gene_list_I)

# Run the Venn diagram function with flipped positions and no labels
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set category names to empty strings to hide labels
    output = TRUE,
    fill = c("blue", "red"),  # Flipped the colors to match the flipped order
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    rotation.degree = 180,  # Rotate the diagram by 180 degrees
    reverse = TRUE  # Reverse the order of categories
)

# Plot the diagram
grid.draw(venn.plot)


In [ ]:
# Loading the DE genes from analysis I and II
gene_list_I <- na.omit(rownames(res2_I_down))  # Remove NAs
gene_list_II <- na.omit(rownames(res2_II_down))  # Remove NAs

# Making a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("list1", "list2"),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = "black",  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
# Loading the DE genes from analysis I and II
gene_list_I <- na.omit(rownames(res2_I_down))  # Remove NAs
gene_list_II <- na.omit(rownames(res2_II_down))  # Remove NAs

# Making a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = NA,  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
# Extract log2FC for the top 50 genes from res2_I_sig and res2_II_sig
# Ensuring only matching rows are selected and handling potential NA values
log2FC_I <- res2_I_sig[match(rownames(top50FilteredData), rownames(res2_I_sig)), "log2FoldChange", drop = FALSE]
log2FC_II <- res2_II_sig[match(rownames(top50FilteredData), rownames(res2_II_sig)), "log2FoldChange", drop = FALSE]


In [ ]:
colnames(log2FC_I)[1] <- "log2FoldChange_I"
colnames(log2FC_II)[1] <- "log2FoldChange_II"
log2FC_I$Gene = rownames(log2FC_I)
log2FC_II$Gene = rownames(log2FC_II)

log2FC_II <- data.frame(log2FC_II)
log2FC_I <- data.frame(log2FC_I)

# Merging the data frames on the 'Gene' column
merged_data = merge(log2FC_I, log2FC_II, by = "Gene", all = TRUE)  # all = TRUE is like a full join



# View the merged data frame
print(merged_data)
rownames(merged_data) <- merged_data$Gene
merged_data$Gene <- NULL

In [ ]:
merged_data$Gene_name <- rownames(merged_data)

In [ ]:
library(ggplot2)
library(reshape2)

# Convert the data from wide to long format
merged_data_long <- melt(merged_data, id.vars = "Gene_name", variable.name = "Condition", value.name = "Log2FoldChange")

# Create a connected scatter plot
p_connected <- ggplot(merged_data_long, aes(x = Condition, y = Log2FoldChange, group = Gene_name)) +
  geom_line(aes(color = Gene_name), size = 1) +  # Connect lines between conditions for each gene
  geom_point(size = 3, aes(color = Gene_name)) +  # Add points for each condition
  theme_minimal() +
  labs(x = "Condition", y = "Log2 Fold Change", title = "Connected Scatter Plot of Log2 Fold Changes")

# Display the plot
print(p_connected)


In [ ]:
# Convert to a suitable format for a lollipop chart
merged_data_wide <- dcast(merged_data_long, Gene_name ~ Condition, value.var = "Log2FoldChange")

# Create a lollipop chart
p_lollipop <- ggplot(merged_data_wide, aes(x = Gene_name)) +
  geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), col = "grey") +
  geom_point(aes(y = log2FoldChange_I), color = 'blue', size = 3) +
  geom_point(aes(y = log2FoldChange_II), color = 'red', size = 3) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(x = "Gene", y = "Log2 Fold Change", title = "Lollipop Chart of Log2 Fold Changes")

# Display the plot
print(p_lollipop)


In [ ]:
library(ggplot2)
library(reshape2)
library(dplyr)

# Assuming merged_data_wide is already created and contains columns: Gene, log2FoldChange_I, log2FoldChange_II

# Create subsets of the data
increased_genes <- merged_data_wide %>% 
  filter(log2FoldChange_I < log2FoldChange_II)

decreased_genes <- merged_data_wide %>% 
  filter(log2FoldChange_I > log2FoldChange_II)


In [ ]:
library(ggplot2)

# Creating the lollipop chart for increased genes with specified modifications
p_lollipop_increased <- ggplot(increased_genes, aes(x = Gene_name)) +
  geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), col = "grey") +
  geom_point(aes(y = log2FoldChange_I), color = 'black', size = 3, stroke = 0.5, shape = 21, fill = '#ff9288') +  # Specify shape and fill for a border
  geom_point(aes(y = log2FoldChange_II), color = 'black', size = 3, stroke = 0.5, shape = 21, fill = '#96cb01') + # Specify shape and fill for a border
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 8),
        axis.text.y = element_text(size = 7),
        axis.line = element_line(size = 0.2),   # Makes axis lines thicker
        panel.grid.major = element_blank(),    # Removes major grid lines
        panel.grid.minor = element_blank(),    # Removes minor grid lines
        plot.title = element_text(size = 14, face = "bold")) +
  labs(x = "Gene", y = "Log2 Fold Change", title = "Increase in Log2 Fold Changes")

# Display the plot
print(p_lollipop_increased)

# Save the plot to a file with specified dimensions
ggsave("Increased_Lollipop_Plot.png", plot = p_lollipop_increased, width = 12, height = 8, dpi = 300)


In [ ]:
library(ggplot2)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_mef"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Creating the lollipop chart
p_lollipop_increased <- ggplot(increased_genes, aes(x = Gene_name)) +
  geom_segment(
    aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II),
    color = "grey"
  ) +
  geom_point(
    aes(y = log2FoldChange_I),
    color = "black",
    size = 4,
    stroke = 0.5,
    shape = 21,
    fill = "#ff9288"
  ) +
  geom_point(
    aes(y = log2FoldChange_II),
    color = "black",
    size = 4,
    stroke = 0.5,
    shape = 21,
    fill = "#96cb01"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 8),
    axis.text.y = element_text(size = 7),
    axis.line = element_line(size = 0.2),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.title = element_text(size = 14, face = "bold")
  ) +
  labs(
    x = "Gene",
    y = "Log2 Fold Change",
    title = "Increase in Log2 Fold Changes"
  )

# Display the plot
print(p_lollipop_increased)

# Save as SVG (vector)
ggsave(
  filename = file.path(out_dir, "Increased_Lollipop_Plot_mef.svg"),
  plot = p_lollipop_increased,
  device = "svg",
  width = 12,
  height = 8,
  units = "in"
)


In [ ]:
library(ggplot2)

# Creating the lollipop chart for increased genes with specified modifications
p_lollipop_decreased <- ggplot(decreased_genes, aes(x = Gene_name)) +
  geom_segment(aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II), col = "grey") +
  geom_point(aes(y = log2FoldChange_I), color = 'black', size = 3, stroke = 0.5, shape = 21, fill = '#ff9288') +  # Specify shape and fill for a border
  geom_point(aes(y = log2FoldChange_II), color = 'black', size = 3, stroke = 0.5, shape = 21, fill = '#96cb01') + # Specify shape and fill for a border
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 12),
        axis.text.y = element_text(size = 7),
        axis.line = element_line(size = 0.2),   # Makes axis lines thicker
        panel.grid.major = element_blank(),    # Removes major grid lines
        panel.grid.minor = element_blank(),    # Removes minor grid lines
        plot.title = element_text(size = 14, face = "bold")) +
  labs(x = "Gene", y = "Log2 Fold Change", title = "Increase in Log2 Fold Changes")

# Display the plot
print(p_lollipop_decreased)

# Save the plot to a file with specified dimensions
ggsave("Decreased_Lollipop_Plot_lmc_v1.png", plot = p_lollipop_decreased, width = 12, height = 8, dpi = 300)


In [ ]:
library(ggplot2)

# Create output directory if it doesn't exist
out_dir <- "all_svg_plots_mef"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

# Creating the lollipop chart for decreased genes
p_lollipop_decreased <- ggplot(decreased_genes, aes(x = Gene_name)) +
  geom_segment(
    aes(xend = Gene_name, y = log2FoldChange_I, yend = log2FoldChange_II),
    color = "grey"
  ) +
  geom_point(
    aes(y = log2FoldChange_I),
    color = "black",
    size = 4,
    stroke = 0.5,
    shape = 21,
    fill = "#ff9288"
  ) +
  geom_point(
    aes(y = log2FoldChange_II),
    color = "black",
    size = 4,
    stroke = 0.5,
    shape = 21,
    fill = "#96cb01"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 12),
    axis.text.y = element_text(size = 7),
    axis.line = element_line(size = 0.2),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.title = element_text(size = 14, face = "bold")
  ) +
  labs(
    x = "Gene",
    y = "Log2 Fold Change",
    title = "Decrease in Log2 Fold Changes"
  )

# Display the plot
print(p_lollipop_decreased)

# Save as SVG (vector)
ggsave(
  filename = file.path(out_dir, "Decreased_Lollipop_Plot_mef.svg"),
  plot = p_lollipop_decreased,
  device = "svg",
  width = 12,
  height = 8,
  units = "in"
)


In [ ]:
# Loading the DE genes from analysis I and II
gene_list_I <- na.omit(rownames(res3_up))  # Remove NAs
gene_list_II <- na.omit(rownames(res3_down))  # Remove NAs

# Making a list of the gene lists
gene_lists <- list(gene_list_I, gene_list_II)

# Run the venn diagram function
venn.plot <- venn.diagram(
    x = gene_lists,
    filename = NULL,
    category.names = c("", ""),  # Set as empty strings
    output = TRUE,
    fill = c("red", "blue"),
    alpha = 0.5,
    fontface = "bold",
    fontsize = 20,
    cat.fontface = "bold",
    cat.fontsize = 20,
    lty = "solid",
    lwd = 2,
    label.col = NA,  # Set this to "black" to show numbers
    category.pos = NULL  # Set to NULL to avoid positioning non-existing labels
)
# Plot the diagram
grid.draw(venn.plot)

In [ ]:
gene_listII <- rownames(res2_II)
entrez_ids <- mapIds(org.Mm.eg.db, keys = gene_listII, column = "ENTREZID", keytype = "SYMBOL")

# Running KEGG enrichment analysis using the Entrez IDs
kegg_results <- enrichKEGG(gene = entrez_ids, organism = 'mmu', keyType = 'kegg')
# Filter to get genes related to NF-κB signaling pathway (mmu04064)
nfkb_genes <- kegg_results$geneID[kegg_results$ID == 'mmu04064']
nfkb_genes_list <- unlist(strsplit(nfkb_genes, split = "/"))
nfkb_gene_names <- mapIds(org.Mm.eg.db, keys = nfkb_genes_list, column = "SYMBOL", keytype = "ENTREZID")
nf_II <- data.frame(nfkb_gene_names)


In [ ]:
colnames(nf_II)[1] <- "Gene_Name"
nf_II$Gene_Name <- toupper(nf_II$Gene_Name)

In [ ]:
filtered_counts <- countData1[countData1$Gene_Name %in% nfkb_gene_names, ]

In [ ]:
rownames(filtered_counts) <- filtered_counts$Gene_Name

In [ ]:
rownames(res2_III) <- toupper(rownames(res2_III))

In [ ]:
rownames(res2_II) <- toupper(rownames(res2_II))

In [ ]:
filtered_data_II <- res2_II[row.names(res2_II) %in% nf_kb, ]

In [ ]:
chk <- data.frame(rownames(res2_III))

In [ ]:
nf_kb <- data.frame(nf_kb)

In [ ]:
colnames(chk)[1] <- "Gene_name"

In [ ]:
colnames(nf_kb)[1] <- "Gene_name"

In [ ]:
chk2 <- merge(nf_kb,chk,by = "Gene_name")

In [ ]:

nf_kb_genes <- intersect(rownames(res2_III), nf_kb$Gene_name)
filteredresdata <- res2_III[nf_kb_genes, ]

filtered_rows <- grep("^(?!GM|MIR|ENSMUS|RPL|RIK).*$", rownames(filteredresdata), perl = TRUE)


In [ ]:
filteredresdata

In [ ]:
# Sort the filtered genes by the absolute value of log2FoldChange in descending order
sorted_genes <- filteredresdata[order(-abs(filteredresdata$log2FoldChange)), ]

# Select the top 50 genes based on this sorting
top_50_genes <- head(sorted_genes, 50)

In [ ]:
top_50_expression_data <- countData1[rownames(top_50_genes), ]

In [ ]:
# Assuming countdata1 is your data frame or matrix
rownames(countData1) <- toupper(rownames(countData1))


In [ ]:
rownames(top_50_genes)

In [ ]:
# Extract log2FoldChange values for the top 50 genes
log2fc_values_top_50 <- top_50_genes$log2FoldChange

# Convert to a matrix for pheatmap
log2fc_matrix_top_50 <- matrix(log2fc_values_top_50, nrow = length(log2fc_values_top_50), ncol = 1, 
                               dimnames = list(rownames(top_50_genes), "Log2FoldChange"))
genes_to_keep <- !grepl("^(Gm|Mir|ENSMUS|Rpl.*|.*Rik|mt-|Malat1)", rownames(log2fc_matrix_top_50))

# Filter both the log2FoldChange values and update row names accordingly
log2fc_values_filtered <- log2fc_values_top_50[genes_to_keep]
row_names_filtered <- rownames(log2fc_matrix_top_50)[genes_to_keep]

# Update the matrix for pheatmap using filtered genes
log2fc_matrix_filtered <- matrix(log2fc_values_filtered, nrow = length(log2fc_values_filtered), ncol = 1, 
                                 dimnames = list(row_names_filtered, "Log2FoldChange"))



color_palette_fc <- colorRampPalette(c("blue", "white", "red"))(100)
# Plot the heatmap with a slimmer appearance by adjusting cellwidth and cellheight
pheatmap(log2fc_matrix_filtered,
         cluster_rows = FALSE, 
         cluster_cols = FALSE, 
         show_rownames = TRUE,
         color = color_palette_fc,
         cellwidth = 50)  # Adjust the cell height if necessary
# Automatically added: save last plot as SVG
ggsave('figure_216.svg', width = 8, height = 6)


In [ ]:
# Load necessary libraries
library(ggplot2)
library(reshape2)  # For melting the matrix

# Convert your matrix to a long format
long_data <- melt(log2fc_matrix_filtered)

ggplot(long_data, aes(x = Var2, y = Var1, fill = value)) +
  geom_tile() +  # Creates the heatmap tiles
  scale_fill_gradient2(low = "blue", high = "red", mid = "yellow", midpoint = 0) +  # Customize your color scale
  coord_fixed(ratio = 0.5) +  # Adjust this value to change tile thickness
  theme_minimal() +  # Use a minimal theme
  theme(axis.text.x = element_text(angle = 45, hjust = 1), # Rotate x-axis labels if needed
        axis.title = element_blank(), # Remove axis titles for a cleaner look
        legend.title = element_blank()) + # Optionally, adjust legend title
  labs(fill = "Log2 FC") # Customize labels as needed

# Automatically added: save last plot as SVG
ggsave('figure_217.svg', width = 8, height = 6)


In [ ]:
lmc_I <- read.csv('LMC_I.csv')
lmc_II <- read.csv('LMC_II.csv')
lmc_III <- read.csv('LMC_III.csv')

In [ ]:
lmc_I <- data.frame(lmc_I)
lmc_II <- data.frame(lmc_II)
lmc_III <- data.frame(lmc_III)

In [ ]:
mef_I <- data.frame(mef_I)
mef_II <- data.frame(mef_II)
mef_III <- data.frame(mef_III)

In [ ]:
mef_I$lfcSE <- NULL
mef_II$lfcSE <- NULL
mef_III$lfcSE <- NULL

mef_I$pvalue <- NULL
mef_II$pvalue <- NULL
mef_III$pvalue <- NULL

mef_I$stat <- NULL
mef_II$stat <- NULL
mef_III$stat <- NULL

In [ ]:
colnames(mef_I)[1] <- "baseMean_I"
colnames(mef_II)[1] <- "baseMean_II"
colnames(mef_III)[1] <- "baseMean_III"

colnames(mef_I)[2] <- "log2Foldchange_I"
colnames(mef_II)[2] <- "log2Foldchange_II"
colnames(mef_III)[2] <- "log2Foldchange_III"

colnames(mef_I)[3] <- "AdjustedPvalue_I"
colnames(mef_II)[3] <- "AdjustedPvalue_II"
colnames(mef_III)[3] <- "AdjustedPvalue_III"

In [ ]:
lmc_I$lfcSE <- NULL
lmc_II$lfcSE <- NULL
lmc_III$lfcSE <- NULL

lmc_I$pvalue <- NULL
lmc_II$pvalue <- NULL
lmc_III$pvalue <- NULL

lmc_I$stat <- NULL
lmc_II$stat <- NULL
lmc_III$stat <- NULL


In [ ]:
colnames(lmc_I)[1] <- "Gene"
colnames(lmc_II)[1] <- "Gene"
colnames(lmc_III)[1] <- "Gene"

colnames(lmc_I)[2] <- "baseMean_I"
colnames(lmc_II)[2] <- "baseMean_II"
colnames(lmc_III)[2] <- "baseMean_III"

colnames(lmc_I)[3] <- "log2Foldchange_I"
colnames(lmc_II)[3] <- "log2Foldchange_II"
colnames(lmc_III)[3] <- "log2Foldchange_III"

colnames(lmc_I)[4] <- "AdjustedPvalue_I"
colnames(lmc_II)[4] <- "AdjustedPvalue_II"
colnames(lmc_III)[4] <- "AdjustedPvalue_III"

In [ ]:
lmc_I_II <- merge(lmc_I,lmc_II,by = "Gene")

In [ ]:
mef_I$Gene <- rownames(mef_I)
mef_II$Gene <- rownames(mef_II)
mef_III$Gene <- rownames(mef_III)

In [ ]:
mef_I_II <- merge(mef_I,mef_II,by = "Gene")

In [ ]:
mef_I_II 

In [ ]:
mef_I_II_filtered <- filter(mef_I_II, AdjustedPvalue_I < 0.05 | AdjustedPvalue_II < 0.05)

In [ ]:
ggplot(mef_I_II_filtered, aes(x = log2Foldchange_I, y = log2Foldchange_II)) +
  geom_point() +
  theme_minimal() +
  labs(x = "Log2 Fold Change I", y = "Log2 Fold Change II", title = "Scatter plot of Log2 Fold Changes in MEF Dataset") +
  geom_smooth(method = "lm", color = "blue", se = FALSE) # Adds a linear regression line without confidence interval
# Automatically added: save last plot as SVG
ggsave('figure_229_mef.svg', width = 8, height = 6)


In [ ]:
library(ggplot2)

p <- ggplot(mef_I_II_filtered, aes(x = log2Foldchange_I, y = log2Foldchange_II)) +
  geom_point() +
  theme_minimal() +
  labs(
    x = "Log2 Fold Change I",
    y = "Log2 Fold Change II",
    title = "Scatter plot of Log2 Fold Changes in LMC Dataset"
  ) +
  geom_smooth(method = "lm", color = "blue", se = FALSE) +
  geom_abline(slope = 1, intercept = 0,
              color = "red",
              linewidth = 1.5,     # use linewidth (newer ggplot2)
              linetype = "longdash")

ggsave("figure_229_mef.svg", plot = p, device = "svg", width = 8, height = 6, units = "in")


In [ ]:
lmc_I_II_filtered <- filter(lmc_I_II, AdjustedPvalue_I < 0.05 | AdjustedPvalue_II < 0.05)

In [ ]:
ggplot(lmc_I_II_filtered, aes(x = log2Foldchange_I, y = log2Foldchange_II)) +
  geom_point() +
  theme_minimal() +
  labs(x = "Log2 Fold Change I", y = "Log2 Fold Change II", title = "Scatter plot of Log2 Fold Changes in LMC Dataset") +
  geom_smooth(method = "lm", color = "blue", se = FALSE) # Adds a linear regression line without confidence interval
# Automatically added: save last plot as SVG
ggsave('figure_229.svg', width = 8, height = 6)


In [ ]:
library(ggplot2)

p <- ggplot(lmc_I_II_filtered, aes(x = log2Foldchange_I, y = log2Foldchange_II)) +
  geom_point() +
  theme_minimal() +
  labs(
    x = "Log2 Fold Change I",
    y = "Log2 Fold Change II",
    title = "Scatter plot of Log2 Fold Changes in LMC Dataset"
  ) +
  geom_smooth(method = "lm", color = "blue", se = FALSE) +
  geom_abline(slope = 1, intercept = 0,
              color = "red",
              linewidth = 1.5,     # use linewidth (newer ggplot2)
              linetype = "longdash")

ggsave("figure_229.svg", plot = p, device = "svg", width = 8, height = 6, units = "in")


In [ ]:
mef_I_II_filtered

In [ ]:
# Assuming 'lmc_I_II_filtered' is correctly set up with 'log2Foldchange_I', 'log2Foldchange_II', and 'Gene' columns.

# Load necessary libraries
library(plotly)

# Define the range for the diagonal line based on your data limits
range_min <- min(c(lmc_I_II_filtered$log2Foldchange_I, lmc_I_II_filtered$log2Foldchange_II))
range_max <- max(c(lmc_I_II_filtered$log2Foldchange_I, lmc_I_II_filtered$log2Foldchange_II))

# Fit a linear model for the regression line
lm_fit <- lm(log2Foldchange_II ~ log2Foldchange_I, data = lmc_I_II_filtered)

# Generate predicted values for the regression line
predict_x <- seq(range_min, range_max, length.out = 100)
predict_y <- predict(lm_fit, newdata = data.frame(log2Foldchange_I = predict_x))

# Create the interactive plot with gene names in hover text
p_interactive <- plot_ly() %>%
  add_trace(data = lmc_I_II_filtered, x = ~log2Foldchange_I, y = ~log2Foldchange_II, type = 'scatter', mode = 'markers',
            hoverinfo = 'text', text = ~paste('Gene:', Gene, '<br>Log2 Fold Change I:', log2Foldchange_I, 
                                              '<br>Log2 Fold Change II:', log2Foldchange_II)) %>%
  #add_trace(x = c(range_min, range_max), y = c(range_min, range_max), type = 'scatter', mode = 'lines',
            #line = list(dash = 'dash', color = 'red'), showlegend = FALSE) %>%
  add_trace(x = predict_x, y = predict_y, type = 'scatter', mode = 'lines',
            line = list(color = 'blue'), showlegend = FALSE) %>%
  layout(title = 'Scatter plot of Log2 Fold Changes in LMC Dataset',
         xaxis = list(title = 'Log2Fold-Change in Wild-type samples (LPS versus Control)'),
         yaxis = list(title = 'Log2Fold-Change in Sp3 knockout samples (LPS versus Control)'),
         hovermode = 'closest')

# Optionally, save the interactive plot as an HTML file
htmlwidgets::saveWidget(p_interactive, file = "interactive_plot_lmc_filtered_I_II_withgenenames.html")



In [ ]:
# Assuming 'lmc_I_II_filtered' is correctly set up with 'log2Foldchange_I', 'log2Foldchange_II', and 'Gene' columns.

# Load necessary libraries
library(plotly)

# Define the range for the diagonal line based on your data limits
range_min <- min(c(lmc_I_II_filtered$log2Foldchange_I, lmc_I_II_filtered$log2Foldchange_II))
range_max <- max(c(lmc_I_II_filtered$log2Foldchange_I, lmc_I_II_filtered$log2Foldchange_II))

# Calculate number of data points above and below the diagonal
above_diagonal <- sum(lmc_I_II_filtered$log2Foldchange_II > lmc_I_II_filtered$log2Foldchange_I)
below_diagonal <- sum(lmc_I_II_filtered$log2Foldchange_II < lmc_I_II_filtered$log2Foldchange_I)

# Fit a linear model for the regression line
lm_fit <- lm(log2Foldchange_II ~ log2Foldchange_I, data = lmc_I_II_filtered)

# Generate predicted values for the regression line
predict_x <- seq(range_min, range_max, length.out = 100)
predict_y <- predict(lm_fit, newdata = data.frame(log2Foldchange_I = predict_x))

# Create the interactive plot with gene names in hover text
p_interactive <- plot_ly() %>%
  add_trace(data = lmc_I_II_filtered, x = ~log2Foldchange_I, y = ~log2Foldchange_II, type = 'scatter', mode = 'markers',
            hoverinfo = 'text', text = ~paste('Gene:', Gene, '<br>Log2 Fold Change I:', log2Foldchange_I, 
                                              '<br>Log2 Fold Change II:', log2Foldchange_II)) %>%
  add_trace(x = c(range_min, range_max), y = c(range_min, range_max), type = 'scatter', mode = 'lines',
            line = list(dash = 'dash', color = 'red'), showlegend = FALSE) %>%
  add_trace(x = predict_x, y = predict_y, type = 'scatter', mode = 'lines',
            line = list(color = 'blue'), showlegend = FALSE) %>%
  layout(title = 'Scatter plot of Log2 Fold Changes in LMC Dataset',
         xaxis = list(title = 'Log2Fold-Change in Wild-type samples (LPS versus Control)'),
         yaxis = list(title = 'Log2Fold-Change in Sp3 knockout samples (LPS versus Control)'),
         hovermode = 'closest',
         annotations = list(
           list(x = 0.95, y = 0.05, xref = 'paper', yref = 'paper', text = paste('Above diagonal:', above_diagonal), showarrow = FALSE, align = 'right'),
           list(x = 0.95, y = 0.00, xref = 'paper', yref = 'paper', text = paste('Below diagonal:', below_diagonal), showarrow = FALSE, align = 'right')
         ))

# Optionally, save the interactive plot as an HTML file
htmlwidgets::saveWidget(p_interactive, file = "interactive_plot_lmc_filtered_I_II_withgenenames_withdiagnol.html")



In [ ]:
sessionInfo()
#display all packages and versions used in the notebook document.